# Compact Benchmark Calculation

9-cell, 15-second CTM benchmark. Kept the model formulas, settings, ramp rules, and assumptions; removed old markdown/report cells, first-step checks, long validation tables, duplicate metadata reloads, and verbose diagnostics.

In [1]:
# 1. Imports, detector IDs, CTM cells, ramp map, metadata

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import datetime as dt

import os
%matplotlib inline

# 1. LOAD METADATA AND SELECT OFFICIAL PISMO CORRIDOR STATIONS
import pandas as pd
from pathlib import Path


# 1. OFFICIAL SELECTED DETECTOR IDS
# 10 mainline + 4 on-ramp + 1 off-ramp = 15 stations
mainline_ids = [
    501015153,  # 4TH ST ML
    501016013,  # 4TH ST ON AREA ML
    501016023,  # PRICE ST ML
    501016031,  # HINDS AVE ML
    501016043,  # BELLO ST ML
    501016053,  # SHELL BEACH RD ML
    501016062,  # MATTIE RD ML
    501016071,  # SPYGLASS DR ML
    501016082,  # AVILA BEACH DR ML
    501016091,  # SAN LUIS BAY DR ML
]

onramp_ids = [
    501016014,  # 4TH ST ON
    501016024,  # PRICE ST ON
    501016063,  # MATTIE RD ON
    501016083,  # AVILA BEACH ON
]

offramp_ids = [
    501016044,  # BELLO ST OFF
]

ids_to_keep = (
    mainline_ids
    + onramp_ids
    + offramp_ids
)

ids_to_keep_str = {
    str(station_id).strip()
    for station_id in ids_to_keep
}

assert len(mainline_ids) == 10
assert len(onramp_ids) == 4
assert len(offramp_ids) == 1
assert len(ids_to_keep) == 15
assert len(ids_to_keep_str) == 15


# 2. OFFICIAL CTM MODEL IDS
cell_ids = [
    f"Cell {i}"
    for i in range(1, 10)
]

segment_ids = [
    f"S{i}"
    for i in range(1, 10)
]

ramp_ids = [
    "u_4th",
    "u_price",
    "u_mattie",
    "u_avila",
]

ramp_name_map = {
    "u_4th": "4TH ST ON",
    "u_price": "PRICE ST ON",
    "u_mattie": "MATTIE RD ON",
    "u_avila": "AVILA BEACH ON",
}



# 3. OFFICIAL CORRECTED RAMP-TO-CELL GEOMETRY.
generic_ramp_cell_map = {
    "u_4th": "Cell 2",
    "u_price": "Cell 2",
    "u_mattie": "Cell 6",
}

merge_ramp_id = "u_avila"
merge_ramp_cell = "Cell 9"

# Full ramp map is still used for metadata, export, queue accounting, and labels.
ramp_cell_map = {
    "u_4th": "Cell 2",
    "u_price": "Cell 2",
    "u_mattie": "Cell 6",
    "u_avila": "Cell 9",
}

ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

assert len(cell_ids) == 9
assert len(segment_ids) == 9
assert len(ramp_ids) == 4
assert set(ramp_cell_map.keys()) == set(ramp_ids)
assert set(ramp_name_map.keys()) == set(ramp_ids)
assert set(generic_ramp_cell_map.keys()) == {
    "u_4th",
    "u_price",
    "u_mattie",
}
assert merge_ramp_id == "u_avila"
assert ramp_cell_map[merge_ramp_id] == merge_ramp_cell
assert all(cell in cell_ids for cell in ramp_cell_map.values())



# 4. LOAD PEMS METADATA

def _candidate_roots():
    roots = []
    start = Path.cwd().resolve()

    for root in [start, *list(start.parents)[:3]]:
        roots.append(root)
        roots.append(root / "HighwayProject")

    clean_roots = []
    seen = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            clean_roots.append(root)

    return clean_roots


def find_input_file(file_name):
    file_name = Path(file_name).name
    requested = Path(file_name)

    search_folders = []
    for root in _candidate_roots():
        search_folders.append(root)
        search_folders.append(root / "District_5_5min_station_data")

    seen = set()
    clean_folders = []
    for folder in search_folders:
        key = str(folder)
        if key not in seen:
            seen.add(key)
            clean_folders.append(folder)

    for folder in clean_folders:
        candidate = folder / file_name
        if candidate.exists():
            return candidate

    pattern = requested.stem + "*" + requested.suffix
    for folder in clean_folders:
        if folder.exists():
            matches = sorted(folder.glob(pattern))
            if matches:
                return matches[0]

    checked = "\n".join(str(folder / file_name) for folder in clean_folders)
    raise FileNotFoundError(
        "Could not find input file: "
        + file_name
        + "\nChecked:\n"
        + checked
    )


metadata_file_path = find_input_file("d05_text_meta_2026_04_28.txt")
print("Loading metadata file:", metadata_file_path)

metadata_raw = pd.read_csv(
    metadata_file_path,
    sep=None,
    engine="python",
    header=None,
    dtype=str,
    skip_blank_lines=True
)

metadata_raw = metadata_raw.apply(
    lambda col: col.astype(str).str.strip()
)

if metadata_raw.shape[1] < 14:
    raise ValueError(
        f"Metadata file has {metadata_raw.shape[1]} columns, expected at least 14."
    )

selected_metadata_clean = pd.DataFrame({
    "station_id": metadata_raw[0],
    "station_name": metadata_raw[13],
    "station_type": metadata_raw[11],
    "freeway": metadata_raw[1],
    "direction": metadata_raw[2],
    "absolute_postmile": metadata_raw[7],
    "length": metadata_raw[10],
    "lanes": metadata_raw[12],
    "latitude": metadata_raw[8],
    "longitude": metadata_raw[9],
})

selected_metadata_clean["station_id"] = (
    selected_metadata_clean["station_id"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(ids_to_keep_str)
].copy()

numeric_cols = [
    "station_id",
    "freeway",
    "absolute_postmile",
    "length",
    "lanes",
    "latitude",
    "longitude",
]

for col in numeric_cols:
    selected_metadata_clean[col] = pd.to_numeric(
        selected_metadata_clean[col],
        errors="coerce"
    )

selected_metadata_clean["station_type"] = (
    selected_metadata_clean["station_type"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean["station_name"] = (
    selected_metadata_clean["station_name"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean = selected_metadata_clean.sort_values(
    [
        "absolute_postmile",
        "station_type",
    ]
).reset_index(drop=True)


# 5. METADATA SAFETY CHECKS
found_station_ids = set(
    selected_metadata_clean["station_id"]
    .dropna()
    .astype(int)
)

expected_station_ids = set(ids_to_keep)

missing_metadata_ids = sorted(
    expected_station_ids - found_station_ids
)

extra_metadata_ids = sorted(
    found_station_ids - expected_station_ids
)

if len(missing_metadata_ids) > 0:
    print("Missing metadata IDs:", missing_metadata_ids)

if len(extra_metadata_ids) > 0:
    print("Extra metadata IDs:", extra_metadata_ids)

if len(selected_metadata_clean) != len(ids_to_keep):
    display(selected_metadata_clean)

    raise ValueError(
        f"Expected {len(ids_to_keep)} selected metadata rows, "
        f"but found {len(selected_metadata_clean)}."
    )

required_metadata_cols = [
    "station_id",
    "station_name",
    "station_type",
    "absolute_postmile",
    "lanes",
]

for col in required_metadata_cols:
    if selected_metadata_clean[col].isna().any():
        bad_rows = selected_metadata_clean[
            selected_metadata_clean[col].isna()
        ].copy()

        print(f"Bad rows for missing column {col}:")
        display(bad_rows)

        raise ValueError(
            f"Missing values found in metadata column: {col}"
        )


# 6. BUILD METADATA DICTIONARIES
station_name_by_id = {
    int(row["station_id"]): str(row["station_name"])
    for _, row in selected_metadata_clean.iterrows()
}

station_type_by_id = {
    int(row["station_id"]): str(row["station_type"])
    for _, row in selected_metadata_clean.iterrows()
}

station_pm = {
    int(row["station_id"]): float(row["absolute_postmile"])
    for _, row in selected_metadata_clean.iterrows()
}

station_lane_count = {
    int(row["station_id"]): int(float(row["lanes"]))
    for _, row in selected_metadata_clean.iterrows()
}

for station_id in ids_to_keep:
    assert station_id in station_name_by_id
    assert station_id in station_type_by_id
    assert station_id in station_pm
    assert station_id in station_lane_count


# 7. STATION TYPE / NAME CHECKS

def normalize_detector_name(station_id):
    return (
        station_name_by_id[station_id]
        .upper()
        .replace(" ", "")
        .replace("-", "")
        .replace("_", "")
    )


def detector_name_indicates_onramp(station_id):
    name = normalize_detector_name(station_id)

    return (
        "ON" in name
        or "ONSB" in name
        or "ONNB" in name
    )


def detector_name_indicates_offramp(station_id):
    name = normalize_detector_name(station_id)

    return (
        "OFF" in name
        or "OFSB" in name
        or "OFNB" in name
    )


for station_id in mainline_ids:
    station_type_upper = station_type_by_id[station_id].upper()

    if station_type_upper != "ML":
        raise ValueError(
            f"Expected mainline station {station_id} to have station_type ML, "
            f"but got station_type={station_type_by_id[station_id]} "
            f"and station_name={station_name_by_id[station_id]}"
        )

for station_id in onramp_ids:
    if not detector_name_indicates_onramp(station_id):
        raise ValueError(
            f"Expected on-ramp station {station_id} name to indicate ON, "
            f"but got station_name={station_name_by_id[station_id]} "
            f"and station_type={station_type_by_id[station_id]}"
        )

for station_id in offramp_ids:
    if not detector_name_indicates_offramp(station_id):
        raise ValueError(
            f"Expected off-ramp station {station_id} name to indicate OFF/OFSB/OFNB, "
            f"but got station_name={station_name_by_id[station_id]} "
            f"and station_type={station_type_by_id[station_id]}"
        )


# 8. OFFICIAL 9-CELL MAINLINE SEGMENT GEOMETRY
mainline_segments = []

for i in range(len(mainline_ids) - 1):
    from_id = mainline_ids[i]
    to_id = mainline_ids[i + 1]

    segment = {
        "segment": segment_ids[i],
        "cell": cell_ids[i],
        "from_id": from_id,
        "to_id": to_id,
        "from_station": station_name_by_id[from_id],
        "to_station": station_name_by_id[to_id],
        "from_postmile": station_pm[from_id],
        "to_postmile": station_pm[to_id],
        "length_miles": station_pm[to_id] - station_pm[from_id],
    }

    if segment["length_miles"] <= 0:
        raise ValueError(
            f"{segment['cell']} has nonpositive length: "
            f"{segment['length_miles']}"
        )

    mainline_segments.append(segment)

assert len(mainline_segments) == 9

mainline_segments_df = pd.DataFrame(mainline_segments)



# 9. DISPLAY OFFICIAL METADATA
print("Official model IDs loaded.")
print("Cells:", cell_ids)
print("Segments:", segment_ids)
print("Ramps:", ramp_ids)
print("Ramp names:", ramp_name_map)

print("Official ramp mapping")
print("generic_ramp_cell_map:", generic_ramp_cell_map)
print("merge_ramp_id:", merge_ramp_id)
print("merge_ramp_cell:", merge_ramp_cell)
print("ramp_cell_map:", ramp_cell_map)

print("Selected Station Metadata")
print("Number of selected stations:", len(selected_metadata_clean))
display(selected_metadata_clean)

print("Official mainline segments")
display(mainline_segments_df)

print("station_pm:")
print(station_pm)

print("station_lane_count:")
print(station_lane_count)

print("PASS: Cell 1 metadata / official geometry loaded cleanly.")

Loading metadata file: C:\Users\User\Desktop\HighwayProject\HighwayProject\d05_text_meta_2026_04_28.txt
Official model IDs loaded.
Cells: ['Cell 1', 'Cell 2', 'Cell 3', 'Cell 4', 'Cell 5', 'Cell 6', 'Cell 7', 'Cell 8', 'Cell 9']
Segments: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Ramps: ['u_4th', 'u_price', 'u_mattie', 'u_avila']
Ramp names: {'u_4th': '4TH ST ON', 'u_price': 'PRICE ST ON', 'u_mattie': 'MATTIE RD ON', 'u_avila': 'AVILA BEACH ON'}
Official ramp mapping
generic_ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6'}
merge_ramp_id: u_avila
merge_ramp_cell: Cell 9
ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6', 'u_avila': 'Cell 9'}
Selected Station Metadata
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,501015153,4TH ST 101 NB EXIT VDS MLSB SB,ML,101,S,188.738,0.506,2,35.133413,-120.616280
1,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,ML,101,S,189.253,0.481,2,35.136794,-120.624363
2,501016014,4TH ST 101 NB ON RAMP VDS ONSB S,OR,101,S,189.254,NaN,1,35.136798,-120.624380
3,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,ML,101,S,189.702,0.475,2,35.138331,-120.632049
4,501016024,PRICE ST EXIT SIGN 101 NB VDS ON,OR,101,S,189.703,NaN,1,35.138334,-120.632066
5,501016031,HINDS AVE 101 SB VDS MLSB SB,ML,101,S,190.204,0.506,2,35.141950,-120.639431
6,501016043,BELLO ST 101 NB VDS MLSB SB,ML,101,S,190.714,0.623,2,35.147275,-120.645623
7,501016044,BELLO ST 101 NB VDS OFSB SB,FR,101,S,190.715,NaN,1,35.147282,-120.645638
8,501016053,SHELL BEACH RD 101 NB VDS MLSB S,ML,101,S,191.451,0.540,2,35.151759,-120.657382
9,501016062,MATTIE RD 101 NB VDS MLSB SB,ML,101,S,191.796,0.935,2,35.154187,-120.662680


Official mainline segments


,segment,cell,from_id,to_id,from_station,to_station,from_postmile,to_postmile,length_miles
0,S1,Cell 1,501015153,501016013,4TH ST 101 NB EXIT VDS MLSB SB,4TH ST 101 NB ON RAMP VDS MLSB S,188.738,189.253,0.515
1,S2,Cell 2,501016013,501016023,4TH ST 101 NB ON RAMP VDS MLSB S,PRICE ST EXIT SIGN 101 NB VDS ML,189.253,189.702,0.449
2,S3,Cell 3,501016023,501016031,PRICE ST EXIT SIGN 101 NB VDS ML,HINDS AVE 101 SB VDS MLSB SB,189.702,190.204,0.502
3,S4,Cell 4,501016031,501016043,HINDS AVE 101 SB VDS MLSB SB,BELLO ST 101 NB VDS MLSB SB,190.204,190.714,0.510
4,S5,Cell 5,501016043,501016053,BELLO ST 101 NB VDS MLSB SB,SHELL BEACH RD 101 NB VDS MLSB S,190.714,191.451,0.737
5,S6,Cell 6,501016053,501016062,SHELL BEACH RD 101 NB VDS MLSB S,MATTIE RD 101 NB VDS MLSB SB,191.451,191.796,0.345
6,S7,Cell 7,501016062,501016071,MATTIE RD 101 NB VDS MLSB SB,SPYGLASS DR 101 SB VDS MLSB SB,191.796,193.322,1.526
7,S8,Cell 8,501016071,501016082,SPYGLASS DR 101 SB VDS MLSB SB,AVILA BEACH DR 101 NB VDS MLSB S,193.322,194.463,1.141
8,S9,Cell 9,501016082,501016091,AVILA BEACH DR 101 NB VDS MLSB S,SAN LUIS BAY DR 101 SB VDS MLSB,194.463,195.520,1.057


station_pm:
{501015153: 188.738, 501016013: 189.253, 501016014: 189.254, 501016023: 189.702, 501016024: 189.703, 501016031: 190.204, 501016043: 190.714, 501016044: 190.715, 501016053: 191.451, 501016062: 191.796, 501016063: 191.797, 501016071: 193.322, 501016082: 194.463, 501016083: 194.464, 501016091: 195.52}
station_lane_count:
{501015153: 2, 501016013: 2, 501016014: 1, 501016023: 2, 501016024: 1, 501016031: 2, 501016043: 2, 501016044: 1, 501016053: 2, 501016062: 2, 501016063: 1, 501016071: 2, 501016082: 3, 501016083: 1, 501016091: 2}
PASS: Cell 1 metadata / official geometry loaded cleanly.


In [2]:
# 2. PeMS 5-minute data, free-flow speeds, benchmark window

# 2. LOAD SINGLE-DAY PEMS 5-MINUTE STATION DATA
# The station file can sit beside the notebook/project files or inside District_5_5min_station_data.
benchmark_date = pd.to_datetime("2026-05-27").date()
benchmark_source_file_requested = "d05_text_station_5min_2026_05_27.txt"
benchmark_file_path = find_input_file(benchmark_source_file_requested)
benchmark_source_file = benchmark_file_path.name

station_files = [
    benchmark_file_path
]

all_days = []

for file_path in station_files:
    file_path = Path(file_path)

    if not file_path.exists():
        print("MISSING:", file_path)
        continue

    print("Loading:", file_path)

    one_day = pd.read_csv(
        file_path,
        header=None
    )

    one_day["source_file"] = file_path.name

    all_days.append(one_day)

if len(all_days) == 0:
    raise FileNotFoundError(
        "No PeMS station files were loaded."
    )

station_data = pd.concat(
    all_days,
    ignore_index=True
)

station_data[0] = pd.to_datetime(
    station_data[0]
)

station_data[1] = pd.to_numeric(
    station_data[1],
    errors="coerce"
).astype("Int64")

filtered_data = station_data[
    station_data[1].isin(ids_to_keep)
].copy()

selected_data = filtered_data[
    [
        0,
        1,
        5,
        9,
        10,
        11,
        "source_file",
    ]
].copy()

selected_data.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow_5min",
    "avg_occupancy",
    "avg_speed",
    "source_file",
]

selected_data["station_id"] = pd.to_numeric(
    selected_data["station_id"],
    errors="coerce"
).astype("Int64")

for col in [
    "total_flow_5min",
    "avg_occupancy",
    "avg_speed",
]:
    selected_data[col] = pd.to_numeric(
        selected_data[col],
        errors="coerce"
    )

selected_data["flow_vph"] = (
    selected_data["total_flow_5min"]
    * 12.0
)

selected_data["date"] = selected_data["timestamp"].dt.date
selected_data["time_of_day"] = selected_data["timestamp"].dt.time
selected_data["hour"] = selected_data["timestamp"].dt.hour

loaded_dates = sorted(
    selected_data["date"].dropna().unique()
)

if loaded_dates != [benchmark_date]:
    raise ValueError(
        f"Expected only benchmark date {benchmark_date}, "
        f"but loaded dates are {loaded_dates}."
    )

print("Selected single-day corridor data")
print("Benchmark source file:", benchmark_source_file)
print("Benchmark date:", benchmark_date)
print("Rows:", len(selected_data))
print("Unique dates:", selected_data["date"].nunique())
print("Unique timestamps:", selected_data["timestamp"].nunique())
print("Unique stations:", selected_data["station_id"].nunique())

display(selected_data.head(20))

# 3. FREE-FLOW WINDOW AND FREE-FLOW SPEEDS
selected_data_midnight = selected_data[
    (selected_data["timestamp"].dt.hour >= 1) &
    (selected_data["timestamp"].dt.hour < 5)
].copy()

print("\nMidnight / Free-Flow Window Data")
print("Time window: 01:00–05:00 across all loaded days")
print("Rows:", len(selected_data_midnight))
print("Unique dates:", selected_data_midnight["date"].nunique())
print("Unique timestamps:", selected_data_midnight["timestamp"].nunique())
print("Unique stations:", selected_data_midnight["station_id"].nunique())

display(selected_data_midnight.head(20))

freeflow_mainline = selected_data_midnight[
    selected_data_midnight["station_id"].isin(mainline_ids)
].copy()

median_speed_by_station = (
    freeflow_mainline
    .groupby("station_id")["avg_speed"]
    .median()
    .reset_index()
)

median_speed_by_station.columns = [
    "station_id",
    "median_speed"
]

print("Station-Level Free-Flow Speeds")
display(median_speed_by_station.round(3))

# 4. OFFICIAL SINGLE-DAY 2-HOUR BENCHMARK WINDOW
print("Finding congestion pattern for selected single benchmark day")

single_day_mainline_data = selected_data[
    selected_data["station_id"].isin(mainline_ids)
].copy()

if single_day_mainline_data.empty:
    raise ValueError(
        "No mainline data found for selected single-day benchmark."
    )

single_day_hour_summary = (
    single_day_mainline_data
    .groupby("hour")
    .agg(
        median_speed=("avg_speed", "median"),
        mean_speed=("avg_speed", "mean"),
        min_speed=("avg_speed", "min"),
        median_flow_vph=("flow_vph", "median"),
        mean_flow_vph=("flow_vph", "mean"),
        pct_speed_below_60=(
            "avg_speed",
            lambda s: (s < 60.0).mean()
        ),
        pct_speed_below_45=(
            "avg_speed",
            lambda s: (s < 45.0).mean()
        ),
        num_rows=("avg_speed", "size"),
        num_time_slots=("time_of_day", "nunique"),
        num_mainline_stations=("station_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        [
            "median_speed",
            "mean_speed",
            "pct_speed_below_45",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
)

print("Single-day hourly congestion summary")
display(single_day_hour_summary.round(3))


# Official benchmark window:
# May 27, 2026, 16:00–18:00
benchmark_profile_type = "single_observed_day_2hour"

benchmark_start_hour = 16
benchmark_end_hour = 18

benchmark_start_time = pd.to_datetime(
    f"{benchmark_start_hour:02d}:00:00"
).time()

benchmark_end_time = pd.to_datetime(
    f"{benchmark_end_hour:02d}:00:00"
).time()

benchmark_window_label = (
    f"{benchmark_start_hour:02d}:00–{benchmark_end_hour:02d}:00"
)

# CTM timing
delta_t = 0.25
steps_per_5min = 20
steps_per_hour = 240

num_steps = int(
    (
        benchmark_end_hour
        - benchmark_start_hour
    )
    * steps_per_hour
)

print("Selected official single-day benchmark")
print("Benchmark date:", benchmark_date)
print("Benchmark window:", benchmark_window_label)
print("Benchmark profile type:", benchmark_profile_type)
print("num_steps:", num_steps)

assert num_steps == 480

# 5. LOAD SINGLE-DAY 2-HOUR BENCHMARK WINDOW DATA
selected_data_afternoon = selected_data[
    (
        selected_data["date"] == benchmark_date
    )
    & (
        selected_data["timestamp"].dt.hour >= benchmark_start_hour
    )
    & (
        selected_data["timestamp"].dt.hour < benchmark_end_hour
    )
].copy()

if selected_data_afternoon.empty:
    raise ValueError(
        f"No rows found for {benchmark_date} {benchmark_window_label}."
    )

print("Single-Day 2-Hour Benchmark Window Data")
print("Benchmark date:", benchmark_date)
print("Benchmark source file:", benchmark_source_file)
print("Time window:", benchmark_window_label)
print("Rows:", len(selected_data_afternoon))
print("Unique dates:", selected_data_afternoon["date"].nunique())
print("Unique timestamps:", selected_data_afternoon["timestamp"].nunique())
print("Unique stations:", selected_data_afternoon["station_id"].nunique())

display(selected_data_afternoon.head(30))


# Since this is a single observed day, each station/time slot should have one row.
single_day_duplicates = (
    selected_data_afternoon
    .groupby(
        [
            "station_id",
            "station_type",
            "time_of_day",
        ]
    )
    .size()
    .reset_index(name="row_count")
)

duplicate_rows = single_day_duplicates[
    single_day_duplicates["row_count"] > 1
].copy()

if len(duplicate_rows) > 0:
    print("Duplicate station/time rows found:")
    display(duplicate_rows.head(50))

    raise ValueError(
        "Single-day benchmark has duplicate station/time rows. "
        "Inspect selected_data_afternoon."
    )


single_day_benchmark_profile = (
    selected_data_afternoon
    .groupby(
        [
            "station_id",
            "station_type",
            "time_of_day",
        ]
    )
    .agg(
        actual_flow_5min=("total_flow_5min", "first"),
        actual_flow_vph=("flow_vph", "first"),
        actual_occupancy=("avg_occupancy", "first"),
        actual_speed=("avg_speed", "first"),
        source_file=("source_file", "first"),
        date=("date", "first"),
    )
    .reset_index()
)

# Backward-compatible aliases.
# Downstream cells expect these names.
# For a single-day benchmark, these are not medians;
# they equal the actual observed values.
typical_pm_profile = single_day_benchmark_profile.copy()

typical_pm_profile["median_flow_5min"] = (
    typical_pm_profile["actual_flow_5min"]
)

typical_pm_profile["median_flow_vph"] = (
    typical_pm_profile["actual_flow_vph"]
)

typical_pm_profile["mean_flow_vph"] = (
    typical_pm_profile["actual_flow_vph"]
)

typical_pm_profile["median_occupancy"] = (
    typical_pm_profile["actual_occupancy"]
)

typical_pm_profile["median_speed"] = (
    typical_pm_profile["actual_speed"]
)

print("Single-day 2-hour benchmark profile")
print("Benchmark date:", benchmark_date)
print("Time window:", benchmark_window_label)
print("Rows:", len(typical_pm_profile))
print("Unique stations:", typical_pm_profile["station_id"].nunique())
print("Unique time slots:", typical_pm_profile["time_of_day"].nunique())

display(typical_pm_profile.head(40))


# Critical coverage checks for CTM
mainline_profile_coverage = typical_pm_profile[
    typical_pm_profile["station_id"].isin(mainline_ids)
].copy()

mainline_time_coverage = (
    mainline_profile_coverage
    .groupby("time_of_day")["station_id"]
    .nunique()
    .reset_index(name="num_mainline_detectors")
)

bad_mainline_time_coverage = mainline_time_coverage[
    mainline_time_coverage["num_mainline_detectors"] != len(mainline_ids)
].copy()

if len(bad_mainline_time_coverage) > 0:
    print("Bad mainline detector coverage by time:")
    display(bad_mainline_time_coverage)

    raise ValueError(
        "Some benchmark time slots do not have all mainline detectors."
    )

expected_5min_rows = int(
    num_steps / steps_per_5min
)

if typical_pm_profile["time_of_day"].nunique() != expected_5min_rows:
    raise ValueError(
        f"Expected {expected_5min_rows} five-minute time slots, "
        f"found {typical_pm_profile['time_of_day'].nunique()}."
    )

print(
    "PASS: single-day 2-hour benchmark profile has full",
    expected_5min_rows,
    "slot mainline coverage."
)

# 6. Segment free-flow speed calculation
def segment_free_flow_speed(upstream_id, downstream_id, median_speed_df):
    v_upstream_series = median_speed_df.loc[
        median_speed_df["station_id"] == upstream_id,
        "median_speed"
    ]

    v_downstream_series = median_speed_df.loc[
        median_speed_df["station_id"] == downstream_id,
        "median_speed"
    ]

    if v_upstream_series.empty:
        raise ValueError(f"Missing median speed for upstream station {upstream_id}")

    if v_downstream_series.empty:
        raise ValueError(f"Missing median speed for downstream station {downstream_id}")

    v_upstream = v_upstream_series.iloc[0]
    v_downstream = v_downstream_series.iloc[0]

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)


mainline_segments = [
    {
        "segment": "S1",
        "from_id": 501015153,
        "to_id": 501016013,
        "from_station": "4TH ST",
        "to_station": "4TH ST ON AREA",
    },
    {
        "segment": "S2",
        "from_id": 501016013,
        "to_id": 501016023,
        "from_station": "4TH ST ON AREA",
        "to_station": "PRICE ST",
    },
    {
        "segment": "S3",
        "from_id": 501016023,
        "to_id": 501016031,
        "from_station": "PRICE ST",
        "to_station": "HINDS AVE",
    },
    {
        "segment": "S4",
        "from_id": 501016031,
        "to_id": 501016043,
        "from_station": "HINDS AVE",
        "to_station": "BELLO ST",
    },
    {
        "segment": "S5",
        "from_id": 501016043,
        "to_id": 501016053,
        "from_station": "BELLO ST",
        "to_station": "SHELL BEACH RD",
    },
    {
        "segment": "S6",
        "from_id": 501016053,
        "to_id": 501016062,
        "from_station": "SHELL BEACH RD",
        "to_station": "MATTIE RD",
    },
    {
        "segment": "S7",
        "from_id": 501016062,
        "to_id": 501016071,
        "from_station": "MATTIE RD",
        "to_station": "SPYGLASS DR",
    },
    {
        "segment": "S8",
        "from_id": 501016071,
        "to_id": 501016082,
        "from_station": "SPYGLASS DR",
        "to_station": "AVILA BEACH DR",
    },
    {
        "segment": "S9",
        "from_id": 501016082,
        "to_id": 501016091,
        "from_station": "AVILA BEACH DR",
        "to_station": "SAN LUIS BAY DR",
    },
]

segment_rows = []

for seg in mainline_segments:
    v_ff = segment_free_flow_speed(
        seg["from_id"],
        seg["to_id"],
        median_speed_by_station
    )

    segment_rows.append({
        "segment": seg["segment"],
        "from_station_id": seg["from_id"],
        "to_station_id": seg["to_id"],
        "from_station": seg["from_station"],
        "to_station": seg["to_station"],
        "v_ff_mph": v_ff
    })
print("Segment Free-Flow-speed")
segment_free_flow_df = pd.DataFrame(segment_rows)
segment_free_flow_df["v_ff_mph"] = segment_free_flow_df["v_ff_mph"].round(3)

display(segment_free_flow_df)

# CTM TIME SETTINGS
# May 27, 16:00–18:00 => 2 hours * 240 steps/hour = 480 steps.
delta_t = 0.25  # minutes, 15 seconds

five_min_intervals_per_hour = 12
steps_per_5min = 20
steps_per_hour = int(60 / delta_t)

benchmark_duration_hours = (
    benchmark_end_hour
    - benchmark_start_hour
)

num_steps = int(
    benchmark_duration_hours
    * steps_per_hour
)

expected_5min_rows = int(
    num_steps
    / steps_per_5min
)

print("CTM time settings")
print("Benchmark window:", benchmark_window_label)
print("Benchmark duration hours:", benchmark_duration_hours)
print("five_min_intervals_per_hour:", five_min_intervals_per_hour)
print("delta_t:", delta_t)
print("steps_per_5min:", steps_per_5min)
print("steps_per_hour:", steps_per_hour)
print("num_steps:", num_steps)
print("expected_5min_rows:", expected_5min_rows)

assert five_min_intervals_per_hour == 12
assert steps_per_5min == 20
assert steps_per_hour == 240
assert num_steps == 480
assert expected_5min_rows == 24

Loading: C:\Users\User\Desktop\HighwayProject\HighwayProject\d05_text_station_5min_2026_05_27.txt
Selected single-day corridor data
Benchmark source file: d05_text_station_5min_2026_05_27.txt
Benchmark date: 2026-05-27
Rows: 4320
Unique dates: 1
Unique timestamps: 288
Unique stations: 15


,timestamp,station_id,station_type,total_flow_5min,avg_occupancy,avg_speed,source_file,flow_vph,date,time_of_day,hour
339,2026-05-27 00:00:00,501015153,ML,16.0,0.0063,68.1,d05_text_station_5min_2026_05_27.txt,192.0,2026-05-27,00:00:00,0
343,2026-05-27 00:00:00,501016013,ML,16.0,0.0063,67.8,d05_text_station_5min_2026_05_27.txt,192.0,2026-05-27,00:00:00,0
344,2026-05-27 00:00:00,501016014,OR,2.0,0.0017,NaN,d05_text_station_5min_2026_05_27.txt,24.0,2026-05-27,00:00:00,0
347,2026-05-27 00:00:00,501016023,ML,18.0,0.0075,67.8,d05_text_station_5min_2026_05_27.txt,216.0,2026-05-27,00:00:00,0
348,2026-05-27 00:00:00,501016024,OR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,00:00:00,0
349,2026-05-27 00:00:00,501016031,ML,19.0,0.0075,68.0,d05_text_station_5min_2026_05_27.txt,228.0,2026-05-27,00:00:00,0
353,2026-05-27 00:00:00,501016043,ML,19.0,0.0074,67.3,d05_text_station_5min_2026_05_27.txt,228.0,2026-05-27,00:00:00,0
354,2026-05-27 00:00:00,501016044,FR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,00:00:00,0
357,2026-05-27 00:00:00,501016053,ML,22.0,0.0093,67.6,d05_text_station_5min_2026_05_27.txt,264.0,2026-05-27,00:00:00,0
359,2026-05-27 00:00:00,501016062,ML,18.0,0.0075,67.1,d05_text_station_5min_2026_05_27.txt,216.0,2026-05-27,00:00:00,0



Midnight / Free-Flow Window Data
Time window: 01:00–05:00 across all loaded days
Rows: 720
Unique dates: 1
Unique timestamps: 48
Unique stations: 15


,timestamp,station_id,station_type,total_flow_5min,avg_occupancy,avg_speed,source_file,flow_vph,date,time_of_day,hour
7383,2026-05-27 01:00:00,501015153,ML,9.0,0.0039,67.9,d05_text_station_5min_2026_05_27.txt,108.0,2026-05-27,01:00:00,1
7387,2026-05-27 01:00:00,501016013,ML,7.0,0.0029,66.7,d05_text_station_5min_2026_05_27.txt,84.0,2026-05-27,01:00:00,1
7388,2026-05-27 01:00:00,501016014,OR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,01:00:00,1
7391,2026-05-27 01:00:00,501016023,ML,6.0,0.0025,67.1,d05_text_station_5min_2026_05_27.txt,72.0,2026-05-27,01:00:00,1
7392,2026-05-27 01:00:00,501016024,OR,1.0,0.0008,NaN,d05_text_station_5min_2026_05_27.txt,12.0,2026-05-27,01:00:00,1
7393,2026-05-27 01:00:00,501016031,ML,7.0,0.0027,66.0,d05_text_station_5min_2026_05_27.txt,84.0,2026-05-27,01:00:00,1
7397,2026-05-27 01:00:00,501016043,ML,11.0,0.0044,65.9,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1
7398,2026-05-27 01:00:00,501016044,FR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,01:00:00,1
7401,2026-05-27 01:00:00,501016053,ML,11.0,0.0048,66.7,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1
7403,2026-05-27 01:00:00,501016062,ML,11.0,0.0046,67.3,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1


Station-Level Free-Flow Speeds


...

Single-day 2-hour benchmark profile
Benchmark date: 2026-05-27
Time window: 16:00–18:00
Rows: 360
Unique stations: 15
Unique time slots: 24


,station_id,station_type,time_of_day,actual_flow_5min,actual_flow_vph,actual_occupancy,actual_speed,source_file,date,median_flow_5min,median_flow_vph,mean_flow_vph,median_occupancy,median_speed
0,501015153,ML,16:00:00,201.0,2412.0,0.0803,66.5,d05_text_station_5min_2026_05_27.txt,2026-05-27,201.0,2412.0,2412.0,0.0803,66.5
1,501015153,ML,16:05:00,194.0,2328.0,0.0791,65.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,194.0,2328.0,2328.0,0.0791,65.9
2,501015153,ML,16:10:00,201.0,2412.0,0.0805,65.8,d05_text_station_5min_2026_05_27.txt,2026-05-27,201.0,2412.0,2412.0,0.0805,65.8
3,501015153,ML,16:15:00,179.0,2148.0,0.0713,66.3,d05_text_station_5min_2026_05_27.txt,2026-05-27,179.0,2148.0,2148.0,0.0713,66.3
4,501015153,ML,16:20:00,190.0,2280.0,0.0766,65.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,190.0,2280.0,2280.0,0.0766,65.9
5,501015153,ML,16:25:00,215.0,2580.0,0.0864,66.2,d05_text_station_5min_2026_05_27.txt,2026-05-27,215.0,2580.0,2580.0,0.0864,66.2
6,501015153,ML,16:30:00,231.0,2772.0,0.0963,64.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,231.0,2772.0,2772.0,0.0963,64.9
7,501015153,ML,16:35:00,248.0,2976.0,0.1058,63.6,d05_text_station_5min_2026_05_27.txt,2026-05-27,248.0,2976.0,2976.0,0.1058,63.6
8,501015153,ML,16:40:00,230.0,2760.0,0.0958,63.5,d05_text_station_5min_2026_05_27.txt,2026-05-27,230.0,2760.0,2760.0,0.0958,63.5
9,501015153,ML,16:45:00,242.0,2904.0,0.1035,62.8,d05_text_station_5min_2026_05_27.txt,2026-05-27,242.0,2904.0,2904.0,0.1035,62.8


PASS: single-day 2-hour benchmark profile has full 24 slot mainline coverage.
Segment Free-Flow-speed


,segment,from_station_id,to_station_id,from_station,to_station,v_ff_mph
0,S1,501015153,501016013,4TH ST,4TH ST ON AREA,66.775
1,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,66.725
2,S3,501016023,501016031,PRICE ST,HINDS AVE,66.825
3,S4,501016031,501016043,HINDS AVE,BELLO ST,66.900
4,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,67.000
5,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,67.025
6,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,66.950
7,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,68.199
8,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,68.147


CTM time settings
Benchmark window: 16:00–18:00
Benchmark duration hours: 2
five_min_intervals_per_hour: 12
delta_t: 0.25
steps_per_5min: 20
steps_per_hour: 240
num_steps: 480
expected_5min_rows: 24


In [3]:
# 3. Doorway capacity, Cell 9 bottleneck, ramp storage, physical storage

#CTM DOORWAY CAPACITY
base_doorway_capacity_rows = []
for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"
    segment = seg["segment"]

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    lanes = station_lane_count[to_id]

    v_ff = float(
        segment_free_flow_df.loc[
            segment_free_flow_df["segment"] == segment,
            "v_ff_mph"
        ].iloc[0]
    )

    if v_ff <= 70:
        base_capacity_per_lane_vph = 1700 + 10 * v_ff
    else:
        base_capacity_per_lane_vph = 2400

    capacity_vph = base_capacity_per_lane_vph * lanes
    capacity_per_5min = capacity_vph / five_min_intervals_per_hour
    capacity_per_15sec = capacity_vph / steps_per_hour

    base_doorway_capacity_rows.append({
        "cell": cell,
        "segment": segment,
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": from_station,
        "to_station": to_station,
        "v_ff_mph": v_ff,
        "lanes": lanes,
        "base_capacity_per_lane_vph": base_capacity_per_lane_vph,
        "capacity_vph": capacity_vph,
        "capacity_per_5min": capacity_per_5min,
        "capacity_per_15sec": capacity_per_15sec,
    })

base_doorway_capacity_df = pd.DataFrame(base_doorway_capacity_rows)

ctm_doorway_capacity_base = {
    row["cell"]: float(row["capacity_per_15sec"])
    for _, row in base_doorway_capacity_df.iterrows()
}

expected_cells = [f"Cell {i}" for i in range(1, 10)]

assert list(ctm_doorway_capacity_base.keys()) == expected_cells
assert all(ctm_doorway_capacity_base[cell] > 0 for cell in expected_cells)

print("Base CTM doorway capacity built successfully.")
display(base_doorway_capacity_df.round(3))

for cell in expected_cells:
    print(cell, round(ctm_doorway_capacity_base[cell], 3))

# DATA-JUSTIFIED SPYGLASS / AVILA / SAN LUIS BAY BOTTLENECK CAPACITY
spyglass_ml_id = 501016071
avila_ml_id = 501016082
san_luis_bay_ml_id = 501016091

raw_pm_data = selected_data_afternoon.copy()

required_cols = [
    "timestamp",
    "station_id",
    "total_flow_5min",
    "avg_speed",
    "date",
    "time_of_day",
]

missing_cols = [
    col
    for col in required_cols
    if col not in raw_pm_data.columns
]

if missing_cols:
    raise KeyError(
        f"selected_data_afternoon is missing columns: {missing_cols}"
    )

raw_pm_data["station_id"] = pd.to_numeric(
    raw_pm_data["station_id"],
    errors="coerce"
).astype("Int64")

raw_pm_data["flow_vph"] = (
    pd.to_numeric(
        raw_pm_data["total_flow_5min"],
        errors="coerce"
    )
    * 12.0
)

raw_pm_data["speed_mph"] = pd.to_numeric(
    raw_pm_data["avg_speed"],
    errors="coerce"
)

bottleneck_station_ids = [
    spyglass_ml_id,
    avila_ml_id,
    san_luis_bay_ml_id,
]

bottleneck_raw = raw_pm_data[
    raw_pm_data["station_id"].isin(bottleneck_station_ids)
].copy()

bottleneck_raw = bottleneck_raw.dropna(
    subset=[
        "flow_vph",
        "speed_mph",
    ]
).copy()

if bottleneck_raw.empty:
    raise ValueError(
        "No raw bottleneck rows found."
    )

flow_pivot = bottleneck_raw.pivot_table(
    index=[
        "date",
        "time_of_day",
    ],
    columns="station_id",
    values="flow_vph",
    aggfunc="first"
)

speed_pivot = bottleneck_raw.pivot_table(
    index=[
        "date",
        "time_of_day",
    ],
    columns="station_id",
    values="speed_mph",
    aggfunc="first"
)

required_station_cols = [
    spyglass_ml_id,
    avila_ml_id,
    san_luis_bay_ml_id,
]

for station_id in required_station_cols:
    if station_id not in flow_pivot.columns:
        raise KeyError(
            f"Missing flow data for station {station_id}"
        )

    if station_id not in speed_pivot.columns:
        raise KeyError(
            f"Missing speed data for station {station_id}"
        )

bottleneck_discharge_raw_df = pd.DataFrame({
    "date": [
        idx[0]
        for idx in flow_pivot.index
    ],
    "time_of_day": [
        idx[1]
        for idx in flow_pivot.index
    ],

    "spyglass_flow_vph": flow_pivot[spyglass_ml_id].values,
    "avila_flow_vph": flow_pivot[avila_ml_id].values,
    "san_luis_bay_flow_vph": flow_pivot[san_luis_bay_ml_id].values,

    "spyglass_speed": speed_pivot[spyglass_ml_id].values,
    "avila_speed": speed_pivot[avila_ml_id].values,
    "san_luis_bay_speed": speed_pivot[san_luis_bay_ml_id].values,
})

bottleneck_discharge_raw_df = bottleneck_discharge_raw_df.dropna().copy()

bottleneck_discharge_raw_df = bottleneck_discharge_raw_df.sort_values(
    [
        "date",
        "time_of_day",
    ]
).reset_index(drop=True)

congestion_speed_threshold_mph = 45.0
downstream_discharge_speed_threshold_mph = 50.0

bottleneck_discharge_raw_df["bottleneck_active"] = (
    (
        bottleneck_discharge_raw_df["spyglass_speed"]
        < congestion_speed_threshold_mph
    )
    & (
        bottleneck_discharge_raw_df["san_luis_bay_speed"]
        >= downstream_discharge_speed_threshold_mph
    )
)

active_discharge_raw = bottleneck_discharge_raw_df[
    bottleneck_discharge_raw_df["bottleneck_active"]
].copy()

if len(active_discharge_raw) == 0:
    raise ValueError(
        "No active bottleneck periods found. "
        "Inspect Spyglass/San Luis Bay speeds or justify a different threshold."
    )

observed_avila_queue_flow_vph_mean = float(
    active_discharge_raw["avila_flow_vph"].mean()
)

observed_avila_queue_flow_vph_p15 = float(
    active_discharge_raw["avila_flow_vph"].quantile(0.15)
)

observed_avila_queue_flow_vph_p85 = float(
    active_discharge_raw["avila_flow_vph"].quantile(0.85)
)

observed_avila_queue_flow_vph_median = float(
    active_discharge_raw["avila_flow_vph"].median()
)

observed_san_luis_bay_discharge_vph_median = float(
    active_discharge_raw["san_luis_bay_flow_vph"].median()
)



# Cell 9 receiving/outflow represents the observed Spyglass -> Avila -> San Luis Bay bottleneck.
#but we used 2400, instead of the observed bc the data is too weak (only 3) , and 2400 matches better to medien than just 3 dataset out of all 24

CAP9_VPH = 2400
ctm_inflow_capacity_official = ctm_doorway_capacity_base.copy()
ctm_inflow_capacity_official["Cell 9"] = (
    CAP9_VPH
    / steps_per_hour
)

ctm_outflow_capacity_official = ctm_doorway_capacity_base.copy()
ctm_outflow_capacity_official["Cell 9"] = (
    CAP9_VPH
    / steps_per_hour
)
# Legacy alias only for tables / old diagnostics.
ctm_doorway_capacity_official = ctm_inflow_capacity_official.copy()

print("Raw active bottleneck periods used for observed capacity")
display(
    active_discharge_raw[
        [
            "date",
            "time_of_day",
            "spyglass_flow_vph",
            "avila_flow_vph",
            "san_luis_bay_flow_vph",
            "spyglass_speed",
            "avila_speed",
            "san_luis_bay_speed",
        ]
    ].round(3)
)

capacity_justification_df = pd.DataFrame([
    {
        "metric": "active_bottleneck_periods",
        "value": len(active_discharge_raw),
        "unit": "5-min periods",
    },
    {
        "metric": "spyglass_congestion_speed_threshold",
        "value": congestion_speed_threshold_mph,
        "unit": "mph",
    },
    {
        "metric": "downstream_discharge_speed_threshold",
        "value": downstream_discharge_speed_threshold_mph,
        "unit": "mph",
    },
    {
        "metric": "observed_avila_queue_flow_median",
        "value": observed_avila_queue_flow_vph_median,
        "unit": "veh/hour",
    },
    {
        "metric": "san_luis_bay_discharge_median",
        "value": observed_san_luis_bay_discharge_vph_median,
        "unit": "veh/hour",
    },
    {
        "metric": "cell9_inflow_bottleneck_capacity",
        "value": ctm_inflow_capacity_official["Cell 9"],
        "unit": "veh/15-sec",
    },
    {
        "metric": "cell9_inflow_bottleneck_capacity_vph",
        "value": ctm_inflow_capacity_official["Cell 9"] * steps_per_hour,
        "unit": "veh/hour",
    },
    {
        "metric": "cell9_outflow_capacity",
        "value": ctm_outflow_capacity_official["Cell 9"],
        "unit": "veh/15-sec",
    },
    {
        "metric": "cell9_outflow_capacity_vph",
        "value": ctm_outflow_capacity_official["Cell 9"] * steps_per_hour,
        "unit": "veh/hour",
    },
])

print("Data-justified split inflow/outflow bottleneck capacity")
display(capacity_justification_df.round(3))

assert list(ctm_inflow_capacity_official.keys()) == cell_ids
assert list(ctm_outflow_capacity_official.keys()) == cell_ids

print("Official CTM doorway capacity built successfully.")

for cell in expected_cells:
    print(
        cell,
        "capacity per 15 sec =",
        round(ctm_doorway_capacity_official[cell], 3),
        "| vph =",
        round(
            ctm_doorway_capacity_official[cell]
            * steps_per_hour,
            1
        )
    )

# RAMP MAXIMUM QUEUE CAPACITY

ave_veh_length = 25  # feet per vehicle
meter_to_feet = 3.28084

ramp_length_m = {
    "4TH ST ON": 154.38,
    "PRICE ST ON": 334.97,
    "MATTIE RD ON": 186.19,
    "AVILA BEACH ON": 438.94,
}

ramp_lane_count = {
    "4TH ST ON": 1,
    "PRICE ST ON": 1,
    "MATTIE RD ON": 1,
    "AVILA BEACH ON": 1,
}

ramp_length_ft = {
    ramp_name: length_m * meter_to_feet
    for ramp_name, length_m in ramp_length_m.items()
}

ramp_max_queue_named = {
    ramp_name: (
        ramp_length_ft[ramp_name]
        * ramp_lane_count[ramp_name]
        / ave_veh_length
    )
    for ramp_name in ramp_length_ft
}

ramp_max_queue_by_u = {
    "u_4th": ramp_max_queue_named["4TH ST ON"],
    "u_price": ramp_max_queue_named["PRICE ST ON"],
    "u_mattie": ramp_max_queue_named["MATTIE RD ON"],
    "u_avila": ramp_max_queue_named["AVILA BEACH ON"],
}

ramp_capacity_df = pd.DataFrame([
    {
        "ramp": ramp_name,
        "ramp_length_m": ramp_length_m[ramp_name],
        "ramp_length_ft": ramp_length_ft[ramp_name],
        "lanes": ramp_lane_count[ramp_name],
        "max_queue_vehicles": ramp_max_queue_named[ramp_name],
    }
    for ramp_name in ramp_length_m
])

print("Ramp Maximum Queue Capacity")
display(ramp_capacity_df.round(3))

assert set(ramp_max_queue_by_u.keys()) == set(ramp_ids)
assert all(ramp_max_queue_by_u[ramp] > 0 for ramp in ramp_ids)

print("PASS: ramp maximum queue capacities built successfully.")

# PHYSICAL STORAGE CAPACITY SETUP FOR 9-CELL CTM
jam_density = 190  # veh / mile / lane

physical_capacity_rows = []

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    length_miles = station_pm[to_id] - station_pm[from_id]

    # Use downstream detector lane count as cell lane count
    lanes = station_lane_count[to_id]

    max_vehicles = jam_density * length_miles * lanes

    physical_capacity_rows.append({
        "cell": cell,
        "segment": seg["segment"],
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": from_station,
        "to_station": to_station,
        "length_miles": length_miles,
        "lanes": lanes,
        "jam_density_veh_mi_lane": jam_density,
        "max_vehicles": max_vehicles,
    })

physical_capacity_df = pd.DataFrame(physical_capacity_rows)

physical_capacity = {
    row["cell"]: float(row["max_vehicles"])
    for _, row in physical_capacity_df.iterrows()
}

cell_length_by_cell = {
    row["cell"]: float(row["length_miles"])
    for _, row in physical_capacity_df.iterrows()
}

cell_lane_count_by_cell = {
    row["cell"]: int(row["lanes"])
    for _, row in physical_capacity_df.iterrows()
}

expected_cells = [f"Cell {i}" for i in range(1, 10)]

assert list(physical_capacity.keys()) == expected_cells
assert list(cell_length_by_cell.keys()) == expected_cells
assert list(cell_lane_count_by_cell.keys()) == expected_cells
assert all(physical_capacity[cell] > 0 for cell in expected_cells)

print("Physical Storage Capacity Setup")
display(physical_capacity_df.round(3))

print("physical_capacity:")
for cell in expected_cells:
    print(cell, round(physical_capacity[cell], 3))

# SAFE THRESHOLD CAPACITY
# Define soft capacity threshold for doorway/storage penalty.
eta = 0.7

safe_threshold_capacity = {
    cell: eta * physical_capacity[cell]
    for cell in cell_ids
}

safe_threshold_capacity_df = pd.DataFrame([
    {
        "cell": cell,
        "physical_capacity": physical_capacity[cell],
        "safe_threshold_capacity": safe_threshold_capacity[cell],
        "eta": eta,
    }
    for cell in cell_ids
])

display(safe_threshold_capacity_df.round(3))

assert list(safe_threshold_capacity.keys()) == cell_ids

Base CTM doorway capacity built successfully.


,cell,segment,from_station_id,to_station_id,from_station,to_station,v_ff_mph,lanes,base_capacity_per_lane_vph,capacity_vph,capacity_per_5min,capacity_per_15sec
0,Cell 1,S1,501015153,501016013,4TH ST,4TH ST ON AREA,66.775,2,2367.75,4735.50,394.625,19.731
1,Cell 2,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,66.725,2,2367.25,4734.50,394.542,19.727
2,Cell 3,S3,501016023,501016031,PRICE ST,HINDS AVE,66.825,2,2368.25,4736.50,394.708,19.735
3,Cell 4,S4,501016031,501016043,HINDS AVE,BELLO ST,66.900,2,2369.00,4738.00,394.833,19.742
4,Cell 5,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,67.000,2,2370.00,4740.00,395.000,19.750
5,Cell 6,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,67.025,2,2370.25,4740.50,395.042,19.752
6,Cell 7,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,66.950,2,2369.50,4739.00,394.917,19.746
7,Cell 8,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,68.199,3,2381.99,7145.97,595.498,29.775
8,Cell 9,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,68.147,2,2381.47,4762.94,396.912,19.846


Cell 1 19.731
Cell 2 19.727
Cell 3 19.735
Cell 4 19.742
Cell 5 19.75
Cell 6 19.752
Cell 7 19.746
Cell 8 29.775
Cell 9 19.846
Raw active bottleneck periods used for observed capacity


,date,time_of_day,spyglass_flow_vph,avila_flow_vph,san_luis_bay_flow_vph,spyglass_speed,avila_speed,san_luis_bay_speed
21,2026-05-27,17:45:00,2700.0,2256.0,3192.0,22.9,13.8,52.3
22,2026-05-27,17:50:00,2460.0,2412.0,2160.0,21.0,13.5,58.5
23,2026-05-27,17:55:00,2736.0,2100.0,2268.0,22.8,20.8,62.1


Data-justified split inflow/outflow bottleneck capacity


...

,ramp,ramp_length_m,ramp_length_ft,lanes,max_queue_vehicles
0,4TH ST ON,154.38,506.496,1,20.260
1,PRICE ST ON,334.97,1098.983,1,43.959
2,MATTIE RD ON,186.19,610.860,1,24.434
3,AVILA BEACH ON,438.94,1440.092,1,57.604


PASS: ramp maximum queue capacities built successfully.
Physical Storage Capacity Setup


,cell,segment,from_station_id,to_station_id,from_station,to_station,length_miles,lanes,jam_density_veh_mi_lane,max_vehicles
0,Cell 1,S1,501015153,501016013,4TH ST,4TH ST ON AREA,0.515,2,190,195.70
1,Cell 2,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,0.449,2,190,170.62
2,Cell 3,S3,501016023,501016031,PRICE ST,HINDS AVE,0.502,2,190,190.76
3,Cell 4,S4,501016031,501016043,HINDS AVE,BELLO ST,0.510,2,190,193.80
4,Cell 5,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,0.737,2,190,280.06
5,Cell 6,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,0.345,2,190,131.10
6,Cell 7,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,1.526,2,190,579.88
7,Cell 8,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,1.141,3,190,650.37
8,Cell 9,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,1.057,2,190,401.66


physical_capacity:
Cell 1 195.7
Cell 2 170.62
Cell 3 190.76
Cell 4 193.8
Cell 5 280.06
Cell 6 131.1
Cell 7 579.88
Cell 8 650.37
Cell 9 401.66


,cell,physical_capacity,safe_threshold_capacity,eta
0,Cell 1,195.70,136.990,0.7
1,Cell 2,170.62,119.434,0.7
2,Cell 3,190.76,133.532,0.7
3,Cell 4,193.80,135.660,0.7
4,Cell 5,280.06,196.042,0.7
5,Cell 6,131.10,91.770,0.7
6,Cell 7,579.88,405.916,0.7
7,Cell 8,650.37,455.259,0.7
8,Cell 9,401.66,281.162,0.7


In [4]:
# 4. CTM input series: boundary demand, ramp commands, exogenous inflow, fixed exits, splits

def build_15sec_series_from_5min_flow(station_id, typical_profile):
    station_5min_data = typical_profile[
        typical_profile["station_id"] == station_id
    ].copy()

    station_5min_data = station_5min_data.sort_values(
        "time_of_day"
    ).reset_index(drop=True)

    expected_5min_rows = int(
        num_steps / steps_per_5min
    )

    if len(station_5min_data) != expected_5min_rows:
        raise ValueError(
            f"Station {station_id} has {len(station_5min_data)} 5-min rows, "
            f"but expected {expected_5min_rows} rows for {benchmark_window_label}."
        )

    station_5min_data["median_flow_5min"] = pd.to_numeric(
        station_5min_data["median_flow_5min"],
        errors="coerce"
    )

    if station_5min_data["median_flow_5min"].isna().any():
        raise ValueError(
            f"Station {station_id} has NaN median_flow_5min values."
        )

    flow_15sec_series = []

    for flow_5min in station_5min_data["median_flow_5min"]:
        flow_15sec = float(flow_5min) / steps_per_5min

        for _ in range(steps_per_5min):
            flow_15sec_series.append(flow_15sec)

    if len(flow_15sec_series) != num_steps:
        raise ValueError(
            f"Station {station_id} produced {len(flow_15sec_series)} steps, "
            f"but expected {num_steps}."
        )

    return flow_15sec_series


# 1. Boundary demand trying to enter Cell 1.
q_in_boundary_series = build_15sec_series_from_5min_flow(
    501015153,
    typical_pm_profile
)


# 2. Controlled ramp benchmark commands.
# In this replay benchmark, the command baseline equals the historical detector release.
arrival_multiplier = 1.2

observed_release_series = {
    "u_4th": build_15sec_series_from_5min_flow(
        501016014,
        typical_pm_profile
    ),
    "u_price": build_15sec_series_from_5min_flow(
        501016024,
        typical_pm_profile
    ),
    "u_mattie": build_15sec_series_from_5min_flow(
        501016063,
        typical_pm_profile
    ),
    "u_avila": build_15sec_series_from_5min_flow(
        501016083,
        typical_pm_profile
    ),
}

commanded_release_series = {
    ramp: [
        float(observed_release_series[ramp][step])
        for step in range(num_steps)
    ]
    for ramp in ramp_ids
}


# 3. Ramp arrivals.
# The stress benchmark uses 1.2 times historical observed ramp release as arriving demand.
ramp_arrival_series = {
    ramp: [
        arrival_multiplier * float(observed_release_series[ramp][step])
        for step in range(num_steps)
    ]
    for ramp in ramp_ids
}


# 4. Mainline detector flow series for detector-balance calibration only.
mainline_flow_series = [
    build_15sec_series_from_5min_flow(
        station_id,
        typical_pm_profile
    )
    for station_id in mainline_ids
]


# 5. Historical controlled on-ramp observations by cell for audit/calibration only.
# These observations are not used as exogenous CTM inflow; actual releases are computed dynamically.
controlled_observed_onramp_series_by_cell = {
    cell: [
        0.0
        for _ in range(num_steps)
    ]
    for cell in cell_ids
}

for ramp, cell in ramp_cell_map.items():
    controlled_observed_onramp_series_by_cell[cell] = [
        controlled_observed_onramp_series_by_cell[cell][step]
        + float(observed_release_series[ramp][step])
        for step in range(num_steps)
    ]


# 6. Detected fixed off-ramp flow by cell.
# Bello is assigned to Cell 5 based on its detector/postmile location relative to the 9-cell geometry.
detected_offramp_cell_map = {
    "bello": "Cell 5"
}

observed_offramp_series = {
    "bello": build_15sec_series_from_5min_flow(
        501016044,
        typical_pm_profile
    )
}

detected_offramp_series_by_cell = {
    cell: [
        0.0
        for _ in range(num_steps)
    ]
    for cell in cell_ids
}

for off_name, cell in detected_offramp_cell_map.items():
    detected_offramp_series_by_cell[cell] = [
        detected_offramp_series_by_cell[cell][step]
        + float(observed_offramp_series[off_name][step])
        for step in range(num_steps)
    ]


# 7. Helper: total vehicles for one station over an hour window.
def total_selected_data_vehicles(
    station_id,
    start_hour,
    end_hour
):
    station_rows = selected_data[
        (
            selected_data["station_id"] == station_id
        )
        & (
            selected_data["date"] == benchmark_date
        )
        & (
            selected_data["timestamp"].dt.hour >= start_hour
        )
        & (
            selected_data["timestamp"].dt.hour < end_hour
        )
    ].copy()

    expected_rows = int(
        (
            end_hour
            - start_hour
        )
        * five_min_intervals_per_hour
    )

    if len(station_rows) != expected_rows:
        raise ValueError(
            f"Station {station_id} has {len(station_rows)} rows "
            f"from {start_hour}:00 to {end_hour}:00, "
            f"but expected {expected_rows}."
        )

    station_rows["total_flow_5min"] = pd.to_numeric(
        station_rows["total_flow_5min"],
        errors="coerce"
    )

    if station_rows["total_flow_5min"].isna().any():
        raise ValueError(
            f"Station {station_id} has NaN total_flow_5min values."
        )

    return float(
        station_rows["total_flow_5min"].sum()
    )


# 8. Detector-balance audit used only to calibrate lateral split/entry assumptions.
def build_segment_balance_audit(
    start_hour,
    end_hour,
    label
):
    rows = []

    duration_hours = (
        end_hour
        - start_hour
    )

    onramp_station_by_ramp = {
        "u_4th": 501016014,
        "u_price": 501016024,
        "u_mattie": 501016063,
        "u_avila": 501016083,
    }

    offramp_station_by_name = {
        "bello": 501016044,
    }

    for i, cell in enumerate(cell_ids):
        upstream_id = mainline_ids[i]
        downstream_id = mainline_ids[i + 1]

        q_up = total_selected_data_vehicles(
            upstream_id,
            start_hour,
            end_hour
        )

        q_down = total_selected_data_vehicles(
            downstream_id,
            start_hour,
            end_hour
        )

        detected_on = 0.0

        for ramp in ramp_ids:
            if ramp_cell_map[ramp] == cell:
                detected_on += total_selected_data_vehicles(
                    onramp_station_by_ramp[ramp],
                    start_hour,
                    end_hour
                )

        detected_off = 0.0

        for off_name, off_cell in detected_offramp_cell_map.items():
            if off_cell == cell:
                detected_off += total_selected_data_vehicles(
                    offramp_station_by_name[off_name],
                    start_hour,
                    end_hour
                )

        net_residual = (
            q_down
            - q_up
            - detected_on
            + detected_off
        )

        inferred_missing_exit = max(
            -net_residual,
            0.0
        )

        inferred_missing_entry = max(
            net_residual,
            0.0
        )

        denominator_for_exit_split = (
            q_up
            + detected_on
        )

        if denominator_for_exit_split > 0:
            exit_split_fraction = (
                inferred_missing_exit
                / denominator_for_exit_split
            )
        else:
            exit_split_fraction = 0.0

        rows.append({
            "audit_window": label,
            "cell": cell,
            "from_detector": upstream_id,
            "to_detector": downstream_id,
            "q_up_veh": q_up,
            "q_down_veh": q_down,
            "detected_controlled_on_veh": detected_on,
            "detected_fixed_off_veh": detected_off,
            "net_residual_veh": net_residual,
            "inferred_missing_exit_veh": inferred_missing_exit,
            "inferred_missing_entry_veh": inferred_missing_entry,
            "missing_exit_vph": (
                inferred_missing_exit
                / duration_hours
            ),
            "missing_entry_vph": (
                inferred_missing_entry
                / duration_hours
            ),
            "exit_split_fraction": exit_split_fraction,
        })

    return pd.DataFrame(rows)


# Calibrated lateral-flow assumptions.
lateral_calibration_start_hour = 13
lateral_calibration_end_hour = 15

BETA_SCALE = 1.5
ENTRY_SCALE = 0.75
MERGE_PRIORITY = 0.3

benchmark_balance_audit_df = build_segment_balance_audit(
    start_hour=benchmark_start_hour,
    end_hour=benchmark_end_hour,
    label=benchmark_window_label
)

lateral_balance_calibration_df = build_segment_balance_audit(
    start_hour=lateral_calibration_start_hour,
    end_hour=lateral_calibration_end_hour,
    label=(
        f"{lateral_calibration_start_hour:02d}:00–"
        f"{lateral_calibration_end_hour:02d}:00"
    )
)

# Backward-compatible alias for older export consumers.
free_flow_balance_audit_df = lateral_balance_calibration_df.copy()

print("Benchmark-window exact detector-balance audit")
display(benchmark_balance_audit_df.round(6))

print("Lateral-flow calibration detector-balance audit")
display(lateral_balance_calibration_df.round(6))


# 9. Build exit split fractions and uncontrolled external entry constants.
raw_exit_split_by_cell = {}
raw_undetected_entry_vph_by_cell = {}

exit_split_by_cell = {}
undetected_entry_vph_by_cell = {}

for _, row in lateral_balance_calibration_df.iterrows():
    cell = row["cell"]

    raw_beta = float(row["exit_split_fraction"])
    raw_entry_vph = float(row["missing_entry_vph"])

    raw_exit_split_by_cell[cell] = raw_beta
    raw_undetected_entry_vph_by_cell[cell] = raw_entry_vph

    exit_split_by_cell[cell] = min(
        0.6,
        max(0.0, raw_beta) * BETA_SCALE
    )

    undetected_entry_vph_by_cell[cell] = (
        max(0.0, raw_entry_vph)
        * ENTRY_SCALE
    )


undetected_entry_per_15sec_by_cell = {
    cell: (
        undetected_entry_vph_by_cell[cell]
        / steps_per_hour
    )
    for cell in cell_ids
}

external_inflow_series = []
fixed_outflow_series = []

for step in range(num_steps):
    external_step = {
        cell: float(undetected_entry_per_15sec_by_cell[cell])
        for cell in cell_ids
    }

    fixed_out_step = {
        cell: float(detected_offramp_series_by_cell[cell][step])
        for cell in cell_ids
    }

    external_inflow_series.append(external_step)
    fixed_outflow_series.append(fixed_out_step)

# Legacy aliases. Important: these now exclude all controlled ramps.
u_in_series = external_inflow_series
f_out_series = fixed_outflow_series

exit_split_df = pd.DataFrame([
    {
        "cell": cell,
        "raw_exit_split_fraction": raw_exit_split_by_cell[cell],
        "calibrated_exit_split_fraction": exit_split_by_cell[cell],
        "raw_undetected_entry_vph": raw_undetected_entry_vph_by_cell[cell],
        "calibrated_undetected_entry_vph": undetected_entry_vph_by_cell[cell],
        "external_entry_per_15sec": undetected_entry_per_15sec_by_cell[cell],
    }
    for cell in cell_ids
])

print("Calibrated lateral-flow model used by CTM dynamics")
display(exit_split_df.round(6))


# 10. Sanity checks.
assert len(q_in_boundary_series) == num_steps
assert len(external_inflow_series) == num_steps
assert len(fixed_outflow_series) == num_steps

for ramp in ramp_ids:
    assert len(observed_release_series[ramp]) == num_steps
    assert len(commanded_release_series[ramp]) == num_steps
    assert len(ramp_arrival_series[ramp]) == num_steps

for step in range(num_steps):
    assert list(external_inflow_series[step].keys()) == cell_ids
    assert list(fixed_outflow_series[step].keys()) == cell_ids

    for ramp in ramp_ids:
        cell = ramp_cell_map[ramp]
        if abs(float(external_inflow_series[step][cell]) - float(undetected_entry_per_15sec_by_cell[cell])) > 1e-12:
            raise AssertionError(
                "Controlled ramp observations leaked into external_inflow_series."
            )

print("Created CTM input series.")
print("q_in_boundary_series:", len(q_in_boundary_series))
print("commanded_release_series:", len(commanded_release_series), "ramps")
print("ramp_arrival_series:", len(ramp_arrival_series), "ramps")
print("external_inflow_series:", len(external_inflow_series), "steps; excludes controlled ramps")
print("fixed_outflow_series:", len(fixed_outflow_series), "steps")
print("PASS: controlled ramp releases are separated from exogenous inflow.")


Benchmark-window exact detector-balance audit


,audit_window,cell,from_detector,to_detector,q_up_veh,q_down_veh,detected_controlled_on_veh,detected_fixed_off_veh,net_residual_veh,inferred_missing_exit_veh,inferred_missing_entry_veh,missing_exit_vph,missing_entry_vph,exit_split_fraction
0,16:00–18:00,Cell 1,501015153,501016013,5199.0,5590.0,0.0,0.0,391.0,0.0,391.0,0.0,195.5,0.000000
1,16:00–18:00,Cell 2,501016013,501016023,5590.0,5766.0,2453.0,0.0,-2277.0,2277.0,0.0,1138.5,0.0,0.283103
2,16:00–18:00,Cell 3,501016023,501016031,5766.0,5338.0,0.0,0.0,-428.0,428.0,0.0,214.0,0.0,0.074228
3,16:00–18:00,Cell 4,501016031,501016043,5338.0,5425.0,0.0,0.0,87.0,0.0,87.0,0.0,43.5,0.000000
4,16:00–18:00,Cell 5,501016043,501016053,5425.0,5868.0,0.0,270.0,713.0,0.0,713.0,0.0,356.5,0.000000
5,16:00–18:00,Cell 6,501016053,501016062,5868.0,5247.0,578.0,0.0,-1199.0,1199.0,0.0,599.5,0.0,0.186007
6,16:00–18:00,Cell 7,501016062,501016071,5247.0,4471.0,0.0,0.0,-776.0,776.0,0.0,388.0,0.0,0.147894
7,16:00–18:00,Cell 8,501016071,501016082,4471.0,3201.0,0.0,0.0,-1270.0,1270.0,0.0,635.0,0.0,0.284053
8,16:00–18:00,Cell 9,501016082,501016091,3201.0,4760.0,625.0,0.0,934.0,0.0,934.0,0.0,467.0,0.000000


Lateral-flow calibration detector-balance audit


,audit_window,cell,from_detector,to_detector,q_up_veh,q_down_veh,detected_controlled_on_veh,detected_fixed_off_veh,net_residual_veh,inferred_missing_exit_veh,inferred_missing_entry_veh,missing_exit_vph,missing_entry_vph,exit_split_fraction
0,13:00–15:00,Cell 1,501015153,501016013,4205.0,4694.0,0.0,0.0,489.0,0.0,489.0,0.0,244.5,0.000000
1,13:00–15:00,Cell 2,501016013,501016023,4694.0,4838.0,1933.0,0.0,-1789.0,1789.0,0.0,894.5,0.0,0.269956
2,13:00–15:00,Cell 3,501016023,501016031,4838.0,4742.0,0.0,0.0,-96.0,96.0,0.0,48.0,0.0,0.019843
3,13:00–15:00,Cell 4,501016031,501016043,4742.0,4862.0,0.0,0.0,120.0,0.0,120.0,0.0,60.0,0.000000
4,13:00–15:00,Cell 5,501016043,501016053,4862.0,5475.0,0.0,368.0,981.0,0.0,981.0,0.0,490.5,0.000000
5,13:00–15:00,Cell 6,501016053,501016062,5475.0,5170.0,305.0,0.0,-610.0,610.0,0.0,305.0,0.0,0.105536
6,13:00–15:00,Cell 7,501016062,501016071,5170.0,4759.0,0.0,0.0,-411.0,411.0,0.0,205.5,0.0,0.079497
7,13:00–15:00,Cell 8,501016071,501016082,4759.0,4704.0,0.0,0.0,-55.0,55.0,0.0,27.5,0.0,0.011557
8,13:00–15:00,Cell 9,501016082,501016091,4704.0,4856.0,443.0,0.0,-291.0,291.0,0.0,145.5,0.0,0.056538


Calibrated lateral-flow model used by CTM dynamics


,cell,raw_exit_split_fraction,calibrated_exit_split_fraction,raw_undetected_entry_vph,calibrated_undetected_entry_vph,external_entry_per_15sec
0,Cell 1,0.000000,0.000000,244.5,183.375,0.764062
1,Cell 2,0.269956,0.404934,0.0,0.000,0.000000
2,Cell 3,0.019843,0.029764,0.0,0.000,0.000000
3,Cell 4,0.000000,0.000000,60.0,45.000,0.187500
4,Cell 5,0.000000,0.000000,490.5,367.875,1.532812
5,Cell 6,0.105536,0.158304,0.0,0.000,0.000000
6,Cell 7,0.079497,0.119246,0.0,0.000,0.000000
7,Cell 8,0.011557,0.017336,0.0,0.000,0.000000
8,Cell 9,0.056538,0.084807,0.0,0.000,0.000000


Created CTM input series.
q_in_boundary_series: 480
commanded_release_series: 4 ramps
ramp_arrival_series: 4 ramps
external_inflow_series: 480 steps; excludes controlled ramps
fixed_outflow_series: 480 steps
PASS: controlled ramp releases are separated from exogenous inflow.


In [5]:
# 5. Initial state, free-flow travel time, movement factors

# PEMS-DERIVED MAINLINE INITIAL STATE
def normalize_time_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, dt.time):
        return value

    if isinstance(value, pd.Timestamp):
        return value.time()

    parsed_value = pd.to_datetime(
        str(value),
        errors="coerce"
    )

    if pd.isna(parsed_value):
        raise ValueError(
            f"Could not parse time_of_day value: {value}"
        )

    return parsed_value.time()


typical_pm_profile = typical_pm_profile.copy()

typical_pm_profile["station_id"] = pd.to_numeric(
    typical_pm_profile["station_id"],
    errors="coerce"
).astype("Int64")

typical_pm_profile["time_of_day_normalized"] = typical_pm_profile[
    "time_of_day"
].apply(normalize_time_value)

if "station_type" in typical_pm_profile.columns:
    typical_pm_profile["station_type"] = (
        typical_pm_profile["station_type"]
        .astype(str)
        .str.strip()
    )


# 1. Find available mainline profile times
mainline_profile_rows = typical_pm_profile[
    typical_pm_profile["station_id"].isin(mainline_ids)
].copy()

if mainline_profile_rows.empty:
    raise ValueError(
        "No rows found in typical_pm_profile for mainline_ids. "
        "Check station_id type or mainline_ids."
    )

available_start_times = sorted(
    [
        t
        for t in mainline_profile_rows["time_of_day_normalized"].dropna().unique()
    ]
)

if len(available_start_times) == 0:
    raise ValueError(
        "No usable time_of_day values found for mainline rows."
    )

if benchmark_start_time not in available_start_times:
    print("Available mainline profile times:")
    print(available_start_times)

    raise ValueError(
        f"Official benchmark_start_time {benchmark_start_time} "
        f"is not available in typical_pm_profile."
    )

print(
    "Using official benchmark initial-state time:",
    benchmark_start_time
)

print("Available mainline profile times:")
print(available_start_times)


# 2. Extract mainline detector data at selected start time
mainline_16pm_state_data = typical_pm_profile[
    (
        typical_pm_profile["station_id"].isin(mainline_ids)
    )
    & (
        typical_pm_profile["time_of_day_normalized"] == benchmark_start_time
    )
].copy()

# Do not require station_type == ML here because mainline_ids already define ML detectors.

for col in [
    "median_flow_5min",
    "median_flow_vph",
    "median_speed",
]:
    mainline_16pm_state_data[col] = pd.to_numeric(
        mainline_16pm_state_data[col],
        errors="coerce"
    )


# 3. Add station metadata
mainline_16pm_state_data["station_name"] = (
    mainline_16pm_state_data["station_id"].map(station_name_by_id)
)

mainline_16pm_state_data["absolute_postmile"] = (
    mainline_16pm_state_data["station_id"].map(station_pm)
)

mainline_16pm_state_data["lanes"] = (
    mainline_16pm_state_data["station_id"].map(station_lane_count)
)

mainline_16pm_state_data = mainline_16pm_state_data.sort_values(
    "absolute_postmile"
).reset_index(drop=True)


# 4. Safety checks
if len(mainline_16pm_state_data) != len(mainline_ids):
    found_ids = set(
        mainline_16pm_state_data["station_id"].dropna().astype(int)
    )

    expected_ids = set(mainline_ids)

    missing_ids = sorted(
        expected_ids - found_ids
    )

    extra_ids = sorted(
        found_ids - expected_ids
    )

    print("Missing mainline IDs:", missing_ids)
    print("Extra mainline IDs:", extra_ids)

    debug_rows = typical_pm_profile[
        typical_pm_profile["station_id"].isin(mainline_ids)
    ][
        [
            "station_id",
            "time_of_day",
            "time_of_day_normalized",
        ]
    ].drop_duplicates().sort_values(
        [
            "station_id",
            "time_of_day_normalized",
        ]
    )

    print("Available rows for expected mainline IDs:")
    display(debug_rows.head(120))

    raise ValueError(
        f"Expected {len(mainline_ids)} mainline detectors at {benchmark_start_time}, "
        f"but found {len(mainline_16pm_state_data)}."
    )


required_cols = [
    "median_flow_5min",
    "median_flow_vph",
    "median_speed",
    "absolute_postmile",
    "lanes",
]

for col in required_cols:
    if mainline_16pm_state_data[col].isna().any():
        raise ValueError(
            f"Missing values in {col} for initial mainline data."
        )

if (
    mainline_16pm_state_data["median_speed"] <= 0
).any():
    raise ValueError(
        "Nonpositive median_speed found in initial mainline detector data."
    )


# 5. Convert flow/speed to detector density
mainline_16pm_state_data["density_veh_per_mile"] = (
    mainline_16pm_state_data["median_flow_vph"]
    / mainline_16pm_state_data["median_speed"]
)


# 6. Build 9 CTM cell geometry
cell_geometry_rows = []

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    cell_start = float(station_pm[from_id])
    cell_end = float(station_pm[to_id])

    cell_midpoint = 0.5 * (
        cell_start
        + cell_end
    )

    cell_length_miles = (
        cell_end
        - cell_start
    )

    if cell_length_miles <= 0:
        raise ValueError(
            f"{cell} has nonpositive length: {cell_length_miles}"
        )

    cell_geometry_rows.append({
        "cell": cell,
        "segment": seg["segment"],
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": seg["from_station"],
        "to_station": seg["to_station"],
        "cell_start_postmile": cell_start,
        "cell_end_postmile": cell_end,
        "cell_midpoint_postmile": cell_midpoint,
        "cell_length_miles": cell_length_miles,
    })

mainline_cell_geometry_df = pd.DataFrame(
    cell_geometry_rows
)


# 7. Interpolate detector density to CTM cell midpoints
detector_postmiles = mainline_16pm_state_data[
    "absolute_postmile"
].to_numpy(dtype=float)

detector_densities = mainline_16pm_state_data[
    "density_veh_per_mile"
].to_numpy(dtype=float)

cell_midpoints = mainline_cell_geometry_df[
    "cell_midpoint_postmile"
].to_numpy(dtype=float)

mainline_cell_geometry_df["interpolated_density_veh_per_mile"] = np.interp(
    cell_midpoints,
    detector_postmiles,
    detector_densities
)

mainline_cell_geometry_df["initial_state_x0"] = (
    mainline_cell_geometry_df["interpolated_density_veh_per_mile"]
    * mainline_cell_geometry_df["cell_length_miles"]
)


# 8. Final benchmark initial state
mainline_initial_state = {
    row["cell"]: float(row["initial_state_x0"])
    for _, row in mainline_cell_geometry_df.iterrows()
}

ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}


# 9. Display derivation tables
mainline_initial_detector_derivation_df = mainline_16pm_state_data[
    [
        "station_id",
        "station_name",
        "absolute_postmile",
        "lanes",
        "median_flow_5min",
        "median_flow_vph",
        "median_speed",
        "density_veh_per_mile",
    ]
].copy()

mainline_initial_state_derivation_df = mainline_cell_geometry_df[
    [
        "cell",
        "segment",
        "from_station",
        "to_station",
        "cell_start_postmile",
        "cell_end_postmile",
        "cell_midpoint_postmile",
        "cell_length_miles",
        "interpolated_density_veh_per_mile",
        "initial_state_x0",
    ]
].copy()

print("PeMS detector-level density derivation")
display(mainline_initial_detector_derivation_df.round(3))

print("PeMS-derived 9-cell initial mainline state")
display(mainline_initial_state_derivation_df.round(3))

print("mainline_initial_state:")
for cell in cell_ids:
    print(
        cell,
        "=",
        round(mainline_initial_state[cell], 3)
    )


# 10. Clean check tables
assert list(mainline_initial_state.keys()) == cell_ids
assert list(physical_capacity.keys()) == cell_ids
assert list(safe_threshold_capacity.keys()) == cell_ids

mainline_initial_state_df = pd.DataFrame([
    {
        "cell": cell,
        "initial_state_x0": mainline_initial_state[cell],
        "safe_threshold_capacity": safe_threshold_capacity[cell],
        "physical_capacity": physical_capacity[cell],
        "x0_over_safe_threshold": (
            mainline_initial_state[cell]
            / safe_threshold_capacity[cell]
        ),
        "x0_over_physical_capacity": (
            mainline_initial_state[cell]
            / physical_capacity[cell]
        ),
    }
    for cell in cell_ids
])

ramp_queue_0_df = pd.DataFrame([
    {
        "ramp": ramp,
        "initial_queue": ramp_queue_0[ramp],
        "max_queue": ramp_max_queue_by_u[ramp],
    }
    for ramp in ramp_ids
])

print("9-Cell Mainline Initial State")
display(mainline_initial_state_df.round(3))

print("Initial Ramp Queue")
display(ramp_queue_0_df.round(3))

# CALCULATE 9-CELL FREE-FLOW TRAVEL TIME
tt_ff_rows = []
tt_ff_min = {}

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"
    segment = seg["segment"]

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    start_postmile = station_pm[from_id]
    end_postmile = station_pm[to_id]

    cell_length_miles = end_postmile - start_postmile

    v_ff_series = segment_free_flow_df.loc[
        segment_free_flow_df["segment"] == segment,
        "v_ff_mph"
    ]

    if v_ff_series.empty:
        raise ValueError(f"Missing free-flow speed for segment {segment}")

    v_ff_mph = float(v_ff_series.iloc[0])

    if v_ff_mph <= 0:
        raise ValueError(f"Invalid free-flow speed for {segment}: {v_ff_mph}")

    TT_ff_hr = cell_length_miles / v_ff_mph
    TT_ff_min_value = TT_ff_hr * 60.0

    tt_ff_min[cell] = TT_ff_min_value

    tt_ff_rows.append({
        "cell": cell,
        "segment": segment,
        "from_station": from_station,
        "to_station": to_station,
        "start_postmile": start_postmile,
        "end_postmile": end_postmile,
        "cell_length_miles": cell_length_miles,
        "v_ff_mph": v_ff_mph,
        "TT_ff_hr": TT_ff_hr,
        "TT_ff_min": TT_ff_min_value,
    })

tt_ff_min_df = pd.DataFrame(tt_ff_rows)

assert list(tt_ff_min.keys()) == cell_ids
assert all(tt_ff_min[cell] > 0 for cell in cell_ids)

print("Calculated 9-Cell Free-Flow Travel Time")
display(tt_ff_min_df.round(5))

print("Calculated tt_ff_min dictionary:")
for cell in cell_ids:
    print(cell, ":", round(tt_ff_min[cell], 5))

print(
    "Total corridor free-flow travel time, min:",
    round(sum(tt_ff_min.values()), 5)
)

# OFFICIAL PHYSICAL MOVEMENT FACTORS
movement_factor_by_cell_official = {
    cell: min(1.0, delta_t / tt_ff_min[cell])
    for cell in cell_ids
}

backward_wave_speed_mph = 15.0

wave_speed_ratio_by_cell_official = {
    cell: min(
        1.0,
        backward_wave_speed_mph
        * (delta_t / 60.0)
        / cell_length_by_cell[cell]
    )
    for cell in cell_ids
}

movement_factor_check_df = pd.DataFrame([
    {
        "cell": cell,
        "tt_ff_min": tt_ff_min[cell],
        "delta_t_min": delta_t,
        "movement_factor": movement_factor_by_cell_official[cell],
        "wave_speed_ratio": wave_speed_ratio_by_cell_official[cell],
    }
    for cell in cell_ids
])

print("Official physical movement factors")
display(movement_factor_check_df.round(4))

assert list(movement_factor_by_cell_official.keys()) == cell_ids
assert list(wave_speed_ratio_by_cell_official.keys()) == cell_ids

assert all(
    0.0 < movement_factor_by_cell_official[cell] <= 1.0
    for cell in cell_ids
)

assert all(
    0.0 < wave_speed_ratio_by_cell_official[cell] <= 1.0
    for cell in cell_ids
)

print("PASS: official movement factors built successfully.")

Using official benchmark initial-state time: 16:00:00
Available mainline profile times:
[datetime.time(16, 0), datetime.time(16, 5), datetime.time(16, 10), datetime.time(16, 15), datetime.time(16, 20), datetime.time(16, 25), datetime.time(16, 30), datetime.time(16, 35), datetime.time(16, 40), datetime.time(16, 45), datetime.time(16, 50), datetime.time(16, 55), datetime.time(17, 0), datetime.time(17, 5), datetime.time(17, 10), datetime.time(17, 15), datetime.time(17, 20), datetime.time(17, 25), datetime.time(17, 30), datetime.time(17, 35), datetime.time(17, 40), datetime.time(17, 45), datetime.time(17, 50), datetime.time(17, 55)]
PeMS detector-level density derivation


,station_id,station_name,absolute_postmile,lanes,median_flow_5min,median_flow_vph,median_speed,density_veh_per_mile
0,501015153,4TH ST 101 NB EXIT VDS MLSB SB,188.738,2,201.0,2412.0,66.5,36.271
1,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,189.253,2,198.0,2376.0,66.9,35.516
2,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,189.702,2,203.0,2436.0,65.9,36.965
3,501016031,HINDS AVE 101 SB VDS MLSB SB,190.204,2,181.0,2172.0,68.9,31.524
4,501016043,BELLO ST 101 NB VDS MLSB SB,190.714,2,176.0,2112.0,67.2,31.429
5,501016053,SHELL BEACH RD 101 NB VDS MLSB S,191.451,2,188.0,2256.0,64.8,34.815
6,501016062,MATTIE RD 101 NB VDS MLSB SB,191.796,2,129.0,1548.0,63.2,24.494
7,501016071,SPYGLASS DR 101 SB VDS MLSB SB,193.322,2,98.0,1176.0,8.4,140.000
8,501016082,AVILA BEACH DR 101 NB VDS MLSB S,194.463,3,53.0,636.0,12.8,49.688
9,501016091,SAN LUIS BAY DR 101 SB VDS MLSB,195.520,2,222.0,2664.0,33.1,80.483


PeMS-derived 9-cell initial mainline state


,cell,segment,from_station,to_station,cell_start_postmile,cell_end_postmile,cell_midpoint_postmile,cell_length_miles,interpolated_density_veh_per_mile,initial_state_x0
0,Cell 1,S1,4TH ST,4TH ST ON AREA,188.738,189.253,188.996,0.515,35.893,18.485
1,Cell 2,S2,4TH ST ON AREA,PRICE ST,189.253,189.702,189.478,0.449,36.240,16.272
2,Cell 3,S3,PRICE ST,HINDS AVE,189.702,190.204,189.953,0.502,34.245,17.191
3,Cell 4,S4,HINDS AVE,BELLO ST,190.204,190.714,190.459,0.510,31.476,16.053
4,Cell 5,S5,BELLO ST,SHELL BEACH RD,190.714,191.451,191.082,0.737,33.122,24.411
5,Cell 6,S6,SHELL BEACH RD,MATTIE RD,191.451,191.796,191.623,0.345,29.654,10.231
6,Cell 7,S7,MATTIE RD,SPYGLASS DR,191.796,193.322,192.559,1.526,82.247,125.509
7,Cell 8,S8,SPYGLASS DR,AVILA BEACH DR,193.322,194.463,193.892,1.141,94.844,108.217
8,Cell 9,S9,AVILA BEACH DR,SAN LUIS BAY DR,194.463,195.520,194.992,1.057,65.085,68.795


mainline_initial_state:
Cell 1 = 18.485
Cell 2 = 16.272
Cell 3 = 17.191
Cell 4 = 16.053
Cell 5 = 24.411
Cell 6 = 10.231
Cell 7 = 125.509
Cell 8 = 108.217
Cell 9 = 68.795
9-Cell Mainline Initial State


...

Calculated 9-Cell Free-Flow Travel Time


,cell,segment,from_station,to_station,start_postmile,end_postmile,cell_length_miles,v_ff_mph,TT_ff_hr,TT_ff_min
0,Cell 1,S1,4TH ST,4TH ST ON AREA,188.738,189.253,0.515,66.775,0.00771,0.46275
1,Cell 2,S2,4TH ST ON AREA,PRICE ST,189.253,189.702,0.449,66.725,0.00673,0.40375
2,Cell 3,S3,PRICE ST,HINDS AVE,189.702,190.204,0.502,66.825,0.00751,0.45073
3,Cell 4,S4,HINDS AVE,BELLO ST,190.204,190.714,0.510,66.900,0.00762,0.45740
4,Cell 5,S5,BELLO ST,SHELL BEACH RD,190.714,191.451,0.737,67.000,0.01100,0.66000
5,Cell 6,S6,SHELL BEACH RD,MATTIE RD,191.451,191.796,0.345,67.025,0.00515,0.30884
6,Cell 7,S7,MATTIE RD,SPYGLASS DR,191.796,193.322,1.526,66.950,0.02279,1.36759
7,Cell 8,S8,SPYGLASS DR,AVILA BEACH DR,193.322,194.463,1.141,68.199,0.01673,1.00383
8,Cell 9,S9,AVILA BEACH DR,SAN LUIS BAY DR,194.463,195.520,1.057,68.147,0.01551,0.93064


Calculated tt_ff_min dictionary:
Cell 1 : 0.46275
Cell 2 : 0.40375
Cell 3 : 0.45073
Cell 4 : 0.4574
Cell 5 : 0.66
Cell 6 : 0.30884
Cell 7 : 1.36759
Cell 8 : 1.00383
Cell 9 : 0.93064
Total corridor free-flow travel time, min: 6.04551
Official physical movement factors


,cell,tt_ff_min,delta_t_min,movement_factor,wave_speed_ratio
0,Cell 1,0.4627,0.25,0.5403,0.1214
1,Cell 2,0.4037,0.25,0.6192,0.1392
2,Cell 3,0.4507,0.25,0.5547,0.1245
3,Cell 4,0.4574,0.25,0.5466,0.1225
4,Cell 5,0.6600,0.25,0.3788,0.0848
5,Cell 6,0.3088,0.25,0.8095,0.1812
6,Cell 7,1.3676,0.25,0.1828,0.0410
7,Cell 8,1.0038,0.25,0.2490,0.0548
8,Cell 9,0.9306,0.25,0.2686,0.0591


PASS: official movement factors built successfully.


In [6]:
# 6. Penalties, ramp queue, receiving-aware CTM step, delay, full simulation
# FAIRNESS PENALTY SETUP
gamma = 1.0

def fairness_penalty_one_step(
    R_next,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
):
    stress_dict = {}

    for u_name, R_t in R_next.items():
        if u_name not in ramp_name_map:
            raise KeyError(f"{u_name} missing from ramp_name_map")

        ramp_name = ramp_name_map[u_name]

        if ramp_name not in ramp_max_queue_named:
            raise KeyError(f"{ramp_name} missing from ramp_max_queue_named")

        R_max_i = float(ramp_max_queue_named[ramp_name])

        if R_max_i <= 0:
            raise ValueError(f"Invalid R_max for {ramp_name}: {R_max_i}")

        raw_stress = float(R_t) / R_max_i
        capped_stress = min(max(raw_stress, 0.0), 1.0)

        stress_dict[ramp_name] = capped_stress

    ramps = list(stress_dict.keys())
    fairness_sum = 0.0

    for i in range(len(ramps)):
        for j in range(i + 1, len(ramps)):
            phi_i = stress_dict[ramps[i]]
            phi_j = stress_dict[ramps[j]]

            fairness_sum += (phi_i - phi_j) ** 2

    L_fair = gamma * fairness_sum

    return stress_dict, fairness_sum, L_fair


# RAMP REQUESTS BEFORE MAINLINE MERGE ACCEPTANCE
def ramp_requested_release_one_step(
    R_current,
    B_current,
    ramp_arrival_step,
    commanded_release_step
):
    available_demand_step = {}
    requested_release_step = {}

    for ramp in ramp_ids:
        if ramp not in R_current:
            raise KeyError(f"{ramp} missing from R_current")

        if ramp not in B_current:
            raise KeyError(f"{ramp} missing from B_current")

        if ramp not in ramp_arrival_step:
            raise KeyError(f"{ramp} missing from ramp_arrival_step")

        if ramp not in commanded_release_step:
            raise KeyError(f"{ramp} missing from commanded_release_step")

        available = (
            max(0.0, float(R_current[ramp]))
            + max(0.0, float(B_current[ramp]))
            + max(0.0, float(ramp_arrival_step[ramp]))
        )

        request = min(
            max(0.0, float(commanded_release_step[ramp])),
            available
        )

        available_demand_step[ramp] = available
        requested_release_step[ramp] = request

    return available_demand_step, requested_release_step


# RAMP QUEUE UPDATE AFTER RECEIVING-AWARE ACTUAL RELEASES
def ramp_next_queue_after_actual_release(
    available_demand_step,
    actual_release_step,
    ramp_max_queue_by_u
):
    R_next = {}
    B_next = {}
    spillback_by_ramp = {}

    for ramp in ramp_ids:
        if ramp not in available_demand_step:
            raise KeyError(f"{ramp} missing from available_demand_step")

        if ramp not in actual_release_step:
            raise KeyError(f"{ramp} missing from actual_release_step")

        if ramp not in ramp_max_queue_by_u:
            raise KeyError(f"{ramp} missing from ramp_max_queue_by_u")

        available = max(0.0, float(available_demand_step[ramp]))
        actual_release = max(0.0, float(actual_release_step[ramp]))
        R_max = float(ramp_max_queue_by_u[ramp])

        if R_max <= 0:
            raise ValueError(f"Invalid R_max for {ramp}: {R_max}")

        if actual_release - available > 1e-9:
            raise ValueError(
                f"Actual release exceeds available demand for {ramp}: "
                f"actual={actual_release}, available={available}"
            )

        waiting_after_release = max(
            0.0,
            available - actual_release
        )

        R_next[ramp] = min(
            waiting_after_release,
            R_max
        )

        B_next[ramp] = max(
            0.0,
            waiting_after_release - R_max
        )

        spillback_by_ramp[ramp] = B_next[ramp]

    return R_next, B_next, spillback_by_ramp


# CAPACITY PENALTY FUNCTION FOR 9-CELL / 15-SEC CTM
lambda_1 = 1.0   # doorway demand pressure penalty weight
lambda_2 = 0.5   # safe threshold penalty weight
lambda_3 = 1.0   # physical capacity penalty weight
lambda_4 = 0.5   # spillback penalty weight

def capacity_penalty_one_step(
    q_in,
    external_inflow_step,
    controllable_onramp_in_step,
    x_next,
    inflow_capacity,
    safe_threshold_capacity,
    physical_capacity,
    spillback_by_ramp,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4
):
    doorway_penalty_by_cell = {}
    safe_threshold_penalty_by_cell = {}
    physical_capacity_penalty_by_cell = {}
    spillback_penalty_by_ramp = {}

    total_doorway_penalty = 0.0
    total_safe_threshold_penalty = 0.0
    total_physical_capacity_penalty = 0.0
    total_spillback_penalty = 0.0

    for cell in cell_ids:
        doorway_demand_pressure = (
            float(q_in.get(cell, 0.0))
            + float(external_inflow_step.get(cell, 0.0))
            + float(controllable_onramp_in_step.get(cell, 0.0))
        )

        doorway_excess = max(
            0.0,
            doorway_demand_pressure - float(inflow_capacity[cell])
        )

        doorway_penalty = lambda_1 * doorway_excess ** 2

        doorway_penalty_by_cell[cell] = doorway_penalty
        total_doorway_penalty += doorway_penalty

        safe_excess = max(
            0.0,
            float(x_next[cell]) - float(safe_threshold_capacity[cell])
        )

        safe_penalty = lambda_2 * safe_excess ** 2

        safe_threshold_penalty_by_cell[cell] = safe_penalty
        total_safe_threshold_penalty += safe_penalty

        physical_excess = max(
            0.0,
            float(x_next[cell]) - float(physical_capacity[cell])
        )

        physical_penalty = lambda_3 * physical_excess ** 2

        physical_capacity_penalty_by_cell[cell] = physical_penalty
        total_physical_capacity_penalty += physical_penalty

    for ramp in ramp_ids:
        spillback_excess = max(
            0.0,
            float(spillback_by_ramp.get(ramp, 0.0))
        )

        spillback_penalty = lambda_4 * spillback_excess ** 2

        spillback_penalty_by_ramp[ramp] = spillback_penalty
        total_spillback_penalty += spillback_penalty

    total_capacity_penalty = (
        total_doorway_penalty
        + total_safe_threshold_penalty
        + total_physical_capacity_penalty
        + total_spillback_penalty
    )

    return {
        "doorway_penalty_by_cell": doorway_penalty_by_cell,
        "safe_threshold_penalty_by_cell": safe_threshold_penalty_by_cell,
        "physical_capacity_penalty_by_cell": physical_capacity_penalty_by_cell,
        "spillback_penalty_by_ramp": spillback_penalty_by_ramp,
        "total_doorway_penalty": total_doorway_penalty,
        "total_safe_threshold_penalty": total_safe_threshold_penalty,
        "total_physical_capacity_penalty": total_physical_capacity_penalty,
        "total_spillback_penalty": total_spillback_penalty,
        "total_capacity_penalty": total_capacity_penalty,
    }


def allocate_ramp_requests_proportionally(
    ramps,
    requested_release_step,
    capacity_available
):
    accepted = {
        ramp: 0.0
        for ramp in ramps
    }

    capacity_available = max(
        0.0,
        float(capacity_available)
    )

    if len(ramps) == 0 or capacity_available <= 0.0:
        return accepted

    total_request = sum(
        max(0.0, float(requested_release_step[ramp]))
        for ramp in ramps
    )

    if total_request <= 0.0:
        return accepted

    if total_request <= capacity_available:
        for ramp in ramps:
            accepted[ramp] = max(
                0.0,
                float(requested_release_step[ramp])
            )

        return accepted

    for ramp in ramps:
        accepted[ramp] = (
            capacity_available
            * max(0.0, float(requested_release_step[ramp]))
            / total_request
        )

    return accepted


def build_controllable_onramp_in_from_actual_release(actual_release_step):
    controllable_onramp_in_step = {
        cell: 0.0
        for cell in cell_ids
    }

    for ramp in ramp_ids:
        if ramp not in actual_release_step:
            raise KeyError(f"{ramp} missing from actual_release_step")

        cell = ramp_cell_map[ramp]

        controllable_onramp_in_step[cell] += float(
            actual_release_step[ramp]
        )

    return controllable_onramp_in_step


# RECEIVING-AWARE 15-SECOND CTM STEP.
# Controlled ramp releases are accepted by the merge/receiving logic before entering the mainline state.
def ctm_15sec_step(
    x_current,
    q_in_boundary_step,
    external_inflow_step,
    fixed_outflow_step,
    requested_release_step,
    inflow_capacity,
    outflow_capacity,
    physical_capacity,
    movement_factor_by_cell,
    wave_speed_ratio_by_cell,
    upstream_boundary_queue=0.0,
    exit_split_by_cell=None,
    merge_priority=None
):
    cells = cell_ids

    if exit_split_by_cell is None:
        exit_split_by_cell = {
            cell: 0.0
            for cell in cells
        }

    if merge_priority is None:
        merge_priority = MERGE_PRIORITY

    assert list(exit_split_by_cell.keys()) == cells
    assert list(external_inflow_step.keys()) == cells
    assert list(fixed_outflow_step.keys()) == cells

    for ramp in ramp_ids:
        if ramp not in requested_release_step:
            raise KeyError(f"{ramp} missing from requested_release_step")

    sending_total = {}
    receiving = {}
    receiving_after_external = {}

    q_out = {
        cell: 0.0
        for cell in cells
    }

    q_in = {
        cell: 0.0
        for cell in cells
    }

    split_exit_flow = {
        cell: 0.0
        for cell in cells
    }

    actual_fixed_f_out = {}
    actual_f_out = {}
    x_next = {}

    actual_release_step = {
        ramp: 0.0
        for ramp in ramp_ids
    }

    merge_diagnostics = {
        "normal_merge_remaining_capacity": {},
        "normal_merge_ramp_requests": {},
        "normal_merge_ramp_acceptance": {},
        "avila_capacity_share": 0.0,
        "avila_request": 0.0,
        "avila_accepted": 0.0,
        "cell8_mainline_demand": 0.0,
        "cell8_mainline_accepted": 0.0,
    }

    # 1. Sending and receiving.
    for cell in cells:
        beta = float(exit_split_by_cell[cell])

        if beta < 0.0 or beta >= 1.0:
            raise ValueError(
                f"exit_split_by_cell[{cell}] must be in [0, 1). Got {beta}."
            )

        sending_total[cell] = min(
            float(movement_factor_by_cell[cell]) * float(x_current[cell]),
            float(outflow_capacity[cell])
        )

        available_storage = max(
            0.0,
            float(physical_capacity[cell]) - float(x_current[cell])
        )

        receiving[cell] = max(
            0.0,
            min(
                float(inflow_capacity[cell]),
                float(wave_speed_ratio_by_cell[cell]) * available_storage
            )
        )

        external_inflow = max(
            0.0,
            float(external_inflow_step[cell])
        )

        receiving_after_external[cell] = max(
            0.0,
            receiving[cell] - external_inflow
        )

    # 2. Boundary inflow with upstream boundary queue.
    boundary_demand = (
        float(upstream_boundary_queue)
        + float(q_in_boundary_step)
    )

    q_in["Cell 1"] = min(
        boundary_demand,
        receiving_after_external["Cell 1"]
    )

    upstream_boundary_queue_next = (
        boundary_demand
        - q_in["Cell 1"]
    )

    upstream_boundary_delay = (
        (
            float(upstream_boundary_queue)
            + float(upstream_boundary_queue_next)
        )
        / 2.0
    ) * delta_t

    # 3. Internal mainline links and ramp merge acceptance.
    normal_ramps_by_cell = {
        cell: []
        for cell in cells
    }

    for ramp, cell in ramp_cell_map.items():
        if ramp != merge_ramp_id:
            normal_ramps_by_cell[cell].append(ramp)

    for i in range(len(cells) - 1):
        cell = cells[i]
        downstream_cell = cells[i + 1]
        beta = float(exit_split_by_cell[cell])

        if 1.0 - beta <= 1e-12:
            mainline_demand = 0.0
        else:
            mainline_demand = (
                1.0
                - beta
            ) * sending_total[cell]

        downstream_capacity = receiving_after_external[downstream_cell]

        if cell == "Cell 8":
            avila_request = max(
                0.0,
                float(requested_release_step[merge_ramp_id])
            )

            avila_capacity_share = max(
                float(merge_priority) * downstream_capacity,
                downstream_capacity - mainline_demand
            )

            avila_capacity_share = max(
                0.0,
                min(
                    downstream_capacity,
                    avila_capacity_share
                )
            )

            avila_accepted = min(
                avila_request,
                avila_capacity_share
            )

            mainline_accepted = min(
                mainline_demand,
                max(
                    0.0,
                    downstream_capacity - avila_accepted
                )
            )

            actual_release_step[merge_ramp_id] = avila_accepted
            q_out[cell] = mainline_accepted

            merge_diagnostics["avila_capacity_share"] = avila_capacity_share
            merge_diagnostics["avila_request"] = avila_request
            merge_diagnostics["avila_accepted"] = avila_accepted
            merge_diagnostics["cell8_mainline_demand"] = mainline_demand
            merge_diagnostics["cell8_mainline_accepted"] = mainline_accepted

        else:
            mainline_accepted = min(
                mainline_demand,
                downstream_capacity
            )

            q_out[cell] = mainline_accepted

            remaining_capacity_for_ramps = max(
                0.0,
                downstream_capacity - mainline_accepted
            )

            ramps_feeding_downstream = normal_ramps_by_cell[downstream_cell]

            ramp_acceptance = allocate_ramp_requests_proportionally(
                ramps=ramps_feeding_downstream,
                requested_release_step=requested_release_step,
                capacity_available=remaining_capacity_for_ramps
            )

            for ramp, accepted in ramp_acceptance.items():
                actual_release_step[ramp] = accepted

            if ramps_feeding_downstream:
                merge_diagnostics["normal_merge_remaining_capacity"][downstream_cell] = remaining_capacity_for_ramps
                merge_diagnostics["normal_merge_ramp_requests"][downstream_cell] = {
                    ramp: float(requested_release_step[ramp])
                    for ramp in ramps_feeding_downstream
                }
                merge_diagnostics["normal_merge_ramp_acceptance"][downstream_cell] = ramp_acceptance.copy()

        if 1.0 - beta <= 1e-12:
            total_leave = 0.0
        else:
            total_leave = q_out[cell] / (1.0 - beta)

        total_leave = min(
            total_leave,
            sending_total[cell]
        )

        split_exit_flow[cell] = (
            beta
            * total_leave
        )

    # 4. Internal inflow identities.
    for i in range(1, len(cells)):
        q_in[cells[i]] = q_out[cells[i - 1]]

    # 5. Terminal Cell 9 outflow.
    beta9 = float(exit_split_by_cell["Cell 9"])
    terminal_leave = sending_total["Cell 9"]

    q_out["Cell 9"] = (
        1.0
        - beta9
    ) * terminal_leave

    split_exit_flow["Cell 9"] = (
        beta9
        * terminal_leave
    )

    # 6. Mainline state update with accepted controlled ramp releases.
    controllable_onramp_in_step = build_controllable_onramp_in_from_actual_release(
        actual_release_step
    )

    for cell in cells:
        external_inflow = max(
            0.0,
            float(external_inflow_step[cell])
        )

        ramp_inflow = max(
            0.0,
            float(controllable_onramp_in_step[cell])
        )

        available_before_fixed_off = max(
            0.0,
            float(x_current[cell])
            + float(q_in[cell])
            + external_inflow
            + ramp_inflow
            - float(q_out[cell])
            - float(split_exit_flow[cell])
        )

        actual_fixed_f_out[cell] = min(
            max(
                0.0,
                float(fixed_outflow_step[cell])
            ),
            available_before_fixed_off
        )

        actual_f_out[cell] = (
            split_exit_flow[cell]
            + actual_fixed_f_out[cell]
        )

        x_next[cell] = (
            float(x_current[cell])
            + float(q_in[cell])
            + external_inflow
            + ramp_inflow
            - float(q_out[cell])
            - float(split_exit_flow[cell])
            - float(actual_fixed_f_out[cell])
        )

        x_next[cell] = max(
            0.0,
            x_next[cell]
        )

    return (
        x_next,
        q_out,
        q_in,
        sending_total,
        receiving,
        receiving_after_external,
        actual_f_out,
        split_exit_flow,
        actual_fixed_f_out,
        actual_release_step,
        controllable_onramp_in_step,
        upstream_boundary_queue_next,
        upstream_boundary_delay,
        merge_diagnostics
    )


def mainline_delay_one_step(
    x_now,
    x_next,
    q_out,
    actual_f_out,
    tt_ff_min,
    delta_t
):
    rows = []

    total_mainline_delay_raw = 0.0
    total_mainline_delay_clipped = 0.0

    for cell in cell_ids:
        if cell not in x_now:
            raise KeyError(f"{cell} missing from x_now")

        if cell not in x_next:
            raise KeyError(f"{cell} missing from x_next")

        if cell not in q_out:
            raise KeyError(f"{cell} missing from q_out")

        if cell not in actual_f_out:
            raise KeyError(f"{cell} missing from actual_f_out")

        if cell not in tt_ff_min:
            raise KeyError(f"{cell} missing from tt_ff_min")

        ttt = (
            (
                float(x_now[cell])
                + float(x_next[cell])
            )
            / 2.0
        ) * float(delta_t)

        ff_term = (
            float(q_out[cell])
            + float(actual_f_out[cell])
        ) * float(tt_ff_min[cell])

        delay_raw = (
            ttt
            - ff_term
        )

        delay_clipped = max(
            delay_raw,
            0.0
        )

        total_mainline_delay_raw += delay_raw
        total_mainline_delay_clipped += delay_clipped

        rows.append({
            "cell": cell,
            "x_now": x_now[cell],
            "x_next": x_next[cell],
            "q_out": q_out[cell],
            "actual_f_out": actual_f_out[cell],
            "TTT_veh_min": ttt,
            "free_flow_component": ff_term,
            "delay_raw": delay_raw,
            "delay_clipped": delay_clipped,
            "mainline_delay_veh_min": delay_clipped,
        })

    mainline_delay_df = pd.DataFrame(rows)

    return (
        mainline_delay_df,
        total_mainline_delay_raw,
        total_mainline_delay_clipped
    )


# LOCAL RAMP DELAY CALCULATION FOR ONE CTM STEP
def ramp_delay_with_cap(
    R_current,
    R_next,
    B_current,
    B_next,
    ramp_arrival_step,
    commanded_release_step,
    requested_release_step,
    actual_release_step,
    delta_t
):
    rows = []
    total_local_delay = 0.0

    for ramp in ramp_ids:
        required_dicts = {
            "R_current": R_current,
            "R_next": R_next,
            "B_current": B_current,
            "B_next": B_next,
            "ramp_arrival_step": ramp_arrival_step,
            "commanded_release_step": commanded_release_step,
            "requested_release_step": requested_release_step,
            "actual_release_step": actual_release_step,
        }

        for name, dictionary in required_dicts.items():
            if ramp not in dictionary:
                raise KeyError(f"{ramp} missing from {name}")

        waiting_current = (
            float(R_current[ramp])
            + float(B_current.get(ramp, 0.0))
        )

        waiting_next = (
            float(R_next[ramp])
            + float(B_next.get(ramp, 0.0))
        )

        waiting_avg = (waiting_current + waiting_next) / 2.0
        local_delay = waiting_avg * delta_t

        total_local_delay += local_delay

        rows.append({
            "ramp": ramp,
            "R_current": R_current[ramp],
            "B_current": B_current.get(ramp, 0.0),
            "arrival": ramp_arrival_step[ramp],
            "commanded_release": commanded_release_step[ramp],
            "requested_release": requested_release_step[ramp],
            "actual_release": actual_release_step[ramp],
            "R_next": R_next[ramp],
            "B_next": B_next.get(ramp, 0.0),
            "waiting_current": waiting_current,
            "waiting_next": waiting_next,
            "waiting_avg": waiting_avg,
            "local_delay_veh_min": local_delay,
        })

    ramp_delay_df = pd.DataFrame(rows)

    return ramp_delay_df, total_local_delay


# OFFICIAL 480-STEP STATE-BASED CTM BENCHMARK SIMULATION
def simulate_state_based_benchmark_480_steps(
    num_steps,
    mainline_initial_state,
    ramp_queue_0,
    q_in_boundary_series,
    commanded_release_series,
    ramp_arrival_series,
    external_inflow_series,
    fixed_outflow_series,
    inflow_capacity,
    outflow_capacity,
    physical_capacity,
    safe_threshold_capacity,
    ramp_name_map,
    ramp_max_queue_named,
    ramp_max_queue_by_u,
    tt_ff_min,
    delta_t,
    gamma,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4,
    movement_factor_by_cell,
    wave_speed_ratio_by_cell,
    exit_split_by_cell,
    external_queue_0=None,
    use_clipped_mainline_delay=True
):
    # 0. Sanity checks.
    assert num_steps == 480, (
        f"Official two-hour benchmark must have 480 steps, got {num_steps}."
    )

    assert list(mainline_initial_state.keys()) == cell_ids
    assert list(inflow_capacity.keys()) == cell_ids
    assert list(outflow_capacity.keys()) == cell_ids
    assert list(physical_capacity.keys()) == cell_ids
    assert list(safe_threshold_capacity.keys()) == cell_ids
    assert list(tt_ff_min.keys()) == cell_ids
    assert list(movement_factor_by_cell.keys()) == cell_ids
    assert list(wave_speed_ratio_by_cell.keys()) == cell_ids
    assert list(exit_split_by_cell.keys()) == cell_ids

    assert set(ramp_queue_0.keys()) == set(ramp_ids)
    assert set(ramp_max_queue_by_u.keys()) == set(ramp_ids)
    assert set(ramp_name_map.keys()) == set(ramp_ids)

    assert merge_ramp_id == "u_avila"
    assert ramp_cell_map[merge_ramp_id] == merge_ramp_cell
    assert merge_ramp_id not in generic_ramp_cell_map

    assert len(q_in_boundary_series) == num_steps
    assert len(external_inflow_series) == num_steps
    assert len(fixed_outflow_series) == num_steps

    for ramp in ramp_ids:
        assert len(commanded_release_series[ramp]) == num_steps
        assert len(ramp_arrival_series[ramp]) == num_steps

    for step in range(num_steps):
        assert list(external_inflow_series[step].keys()) == cell_ids
        assert list(fixed_outflow_series[step].keys()) == cell_ids

    # 1. Initial states.
    x_current = mainline_initial_state.copy()
    R_current = ramp_queue_0.copy()
    upstream_boundary_queue_current = 0.0

    if external_queue_0 is None:
        B_current = {
            ramp: 0.0
            for ramp in ramp_ids
        }
    else:
        assert set(external_queue_0.keys()) == set(ramp_ids)
        B_current = external_queue_0.copy()

    # 2. History container.
    history = {
        "step": [],
        "x": [],
        "R": [],
        "B": [],
        "q_in": [],
        "q_out": [],
        "external_inflow": [],
        "fixed_outflow": [],
        "actual_f_out": [],
        "split_exit_flow": [],
        "actual_fixed_f_out": [],
        "commanded_release": [],
        "requested_release": [],
        "actual_release": [],
        "ramp_arrival": [],
        "ramp_available_demand": [],
        "spillback": [],
        "controllable_onramp_in": [],
        "sending": [],
        "receiving": [],
        "receiving_after_external": [],
        "merge_diagnostics": [],

        # Avila diagnostics
        "avila_available_demand": [],
        "avila_commanded_release": [],
        "avila_release_request": [],
        "avila_release_demand": [],
        "avila_accepted": [],

        "upstream_boundary_queue": [],
        "upstream_boundary_delay": [],
        "mainline_delay": [],
        "mainline_delay_raw": [],
        "mainline_delay_clipped": [],
        "local_delay": [],
        "fairness_penalty": [],
        "doorway_penalty": [],
        "safe_penalty": [],
        "physical_penalty": [],
        "spillback_penalty": [],
        "capacity_penalty": [],
        "total_objective": [],
    }

    # 3. Simulation loop.
    for step in range(num_steps):
        q_in_boundary_step = float(
            q_in_boundary_series[step]
        )

        commanded_release_step = {
            ramp: float(commanded_release_series[ramp][step])
            for ramp in ramp_ids
        }

        ramp_arrival_step = {
            ramp: float(ramp_arrival_series[ramp][step])
            for ramp in ramp_ids
        }

        external_inflow_step = external_inflow_series[step]
        fixed_outflow_step = fixed_outflow_series[step]

        (
            available_demand_step,
            requested_release_step,
        ) = ramp_requested_release_one_step(
            R_current=R_current,
            B_current=B_current,
            ramp_arrival_step=ramp_arrival_step,
            commanded_release_step=commanded_release_step
        )

        (
            x_next,
            q_out,
            q_in,
            sending,
            receiving,
            receiving_after_external,
            actual_f_out,
            split_exit_flow,
            actual_fixed_f_out,
            actual_release_step,
            controllable_onramp_in_step,
            upstream_boundary_queue_next,
            upstream_boundary_delay,
            merge_diagnostics,
        ) = ctm_15sec_step(
            x_current=x_current,
            q_in_boundary_step=q_in_boundary_step,
            external_inflow_step=external_inflow_step,
            fixed_outflow_step=fixed_outflow_step,
            requested_release_step=requested_release_step,
            inflow_capacity=inflow_capacity,
            outflow_capacity=outflow_capacity,
            physical_capacity=physical_capacity,
            movement_factor_by_cell=movement_factor_by_cell,
            wave_speed_ratio_by_cell=wave_speed_ratio_by_cell,
            upstream_boundary_queue=upstream_boundary_queue_current,
            exit_split_by_cell=exit_split_by_cell,
            merge_priority=MERGE_PRIORITY
        )

        (
            R_next,
            B_next,
            spillback_by_ramp,
        ) = ramp_next_queue_after_actual_release(
            available_demand_step=available_demand_step,
            actual_release_step=actual_release_step,
            ramp_max_queue_by_u=ramp_max_queue_by_u
        )

        for ramp in ramp_ids:
            if actual_release_step[ramp] - requested_release_step[ramp] > 1e-9:
                raise AssertionError(
                    f"Actual release exceeds requested release for {ramp}."
                )

        # Local ramp delay.
        _, total_local_delay = ramp_delay_with_cap(
            R_current=R_current,
            R_next=R_next,
            B_current=B_current,
            B_next=B_next,
            ramp_arrival_step=ramp_arrival_step,
            commanded_release_step=commanded_release_step,
            requested_release_step=requested_release_step,
            actual_release_step=actual_release_step,
            delta_t=delta_t
        )

        # Fairness penalty.
        _, _, L_fair = fairness_penalty_one_step(
            R_next=R_next,
            ramp_name_map=ramp_name_map,
            ramp_max_queue_named=ramp_max_queue_named,
            gamma=gamma
        )

        # State-based mainline delay.
        (
            _,
            total_mainline_delay_raw,
            total_mainline_delay_clipped,
        ) = mainline_delay_one_step(
            x_now=x_current,
            x_next=x_next,
            q_out=q_out,
            actual_f_out=actual_f_out,
            tt_ff_min=tt_ff_min,
            delta_t=delta_t
        )

        if use_clipped_mainline_delay:
            total_mainline_delay = (
                total_mainline_delay_clipped
                + upstream_boundary_delay
            )
        else:
            total_mainline_delay = (
                total_mainline_delay_raw
                + upstream_boundary_delay
            )

        # Capacity penalties.
        capacity_info = capacity_penalty_one_step(
            q_in=q_in,
            external_inflow_step=external_inflow_step,
            controllable_onramp_in_step=controllable_onramp_in_step,
            x_next=x_next,
            inflow_capacity=inflow_capacity,
            safe_threshold_capacity=safe_threshold_capacity,
            physical_capacity=physical_capacity,
            spillback_by_ramp=spillback_by_ramp,
            lambda_1=lambda_1,
            lambda_2=lambda_2,
            lambda_3=lambda_3,
            lambda_4=lambda_4
        )

        # Total objective.
        total_objective = (
            total_mainline_delay
            + total_local_delay
            + L_fair
            + capacity_info["total_capacity_penalty"]
        )

        # Store results.
        history["step"].append(step)
        history["x"].append(x_next.copy())
        history["R"].append(R_next.copy())
        history["B"].append(B_next.copy())
        history["q_in"].append(q_in.copy())
        history["q_out"].append(q_out.copy())
        history["external_inflow"].append(external_inflow_step.copy())
        history["fixed_outflow"].append(fixed_outflow_step.copy())
        history["actual_f_out"].append(actual_f_out.copy())
        history["split_exit_flow"].append(split_exit_flow.copy())
        history["actual_fixed_f_out"].append(actual_fixed_f_out.copy())
        history["commanded_release"].append(commanded_release_step.copy())
        history["requested_release"].append(requested_release_step.copy())
        history["actual_release"].append(actual_release_step.copy())
        history["ramp_arrival"].append(ramp_arrival_step.copy())
        history["ramp_available_demand"].append(available_demand_step.copy())
        history["spillback"].append(spillback_by_ramp.copy())
        history["controllable_onramp_in"].append(controllable_onramp_in_step.copy())
        history["sending"].append(sending.copy())
        history["receiving"].append(receiving.copy())
        history["receiving_after_external"].append(receiving_after_external.copy())
        history["merge_diagnostics"].append(merge_diagnostics.copy())

        history["avila_available_demand"].append(
            float(available_demand_step[merge_ramp_id])
        )

        history["avila_commanded_release"].append(
            float(commanded_release_step[merge_ramp_id])
        )

        history["avila_release_request"].append(
            float(requested_release_step[merge_ramp_id])
        )

        history["avila_release_demand"].append(
            float(requested_release_step[merge_ramp_id])
        )

        history["avila_accepted"].append(
            float(actual_release_step[merge_ramp_id])
        )

        history["upstream_boundary_queue"].append(
            upstream_boundary_queue_next
        )

        history["upstream_boundary_delay"].append(
            upstream_boundary_delay
        )

        history["mainline_delay"].append(
            total_mainline_delay
        )

        history["mainline_delay_raw"].append(
            total_mainline_delay_raw
        )

        history["mainline_delay_clipped"].append(
            total_mainline_delay_clipped
        )

        history["local_delay"].append(
            total_local_delay
        )

        history["fairness_penalty"].append(
            L_fair
        )

        history["doorway_penalty"].append(
            capacity_info["total_doorway_penalty"]
        )

        history["safe_penalty"].append(
            capacity_info["total_safe_threshold_penalty"]
        )

        history["physical_penalty"].append(
            capacity_info["total_physical_capacity_penalty"]
        )

        history["spillback_penalty"].append(
            capacity_info["total_spillback_penalty"]
        )

        history["capacity_penalty"].append(
            capacity_info["total_capacity_penalty"]
        )

        history["total_objective"].append(
            total_objective
        )

        # Move to next step.
        x_current = x_next.copy()
        R_current = R_next.copy()
        B_current = B_next.copy()
        upstream_boundary_queue_current = upstream_boundary_queue_next

    # 4. Final states.
    history["R_final"] = R_current.copy()
    history["B_final"] = B_current.copy()
    history["x_final"] = x_current.copy()
    history["upstream_boundary_queue_final"] = upstream_boundary_queue_current

    assert len(history["step"]) == num_steps

    return history


In [7]:
# 7. Run benchmark, validate, save export, show compact summary
# EXPORT SOURCE-OF-TRUTH BENCHMARK INPUTS FOR ADMM-MPC

import pickle
import pandas as pd

# 0. Explicit settings used by both benchmark and ADMM
use_clipped_mainline_delay = True

if external_queue_0 is None:
    external_queue_0_export = {
        ramp: 0.0
        for ramp in ramp_ids
    }
else:
    external_queue_0_export = external_queue_0.copy()


# 1. Official benchmark re-run
official_benchmark_history = simulate_state_based_benchmark_480_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,
    q_in_boundary_series=q_in_boundary_series,
    commanded_release_series=commanded_release_series,
    ramp_arrival_series=ramp_arrival_series,
    external_inflow_series=external_inflow_series,
    fixed_outflow_series=fixed_outflow_series,

    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,

    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,

    tt_ff_min=tt_ff_min,
    delta_t=delta_t,

    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4,

    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    exit_split_by_cell=exit_split_by_cell,
    external_queue_0=external_queue_0_export,
    use_clipped_mainline_delay=use_clipped_mainline_delay
)


# 2. Official benchmark totals
official_totals = {
    "mainline_delay": sum(official_benchmark_history["mainline_delay"]),
    "mainline_delay_raw": sum(official_benchmark_history["mainline_delay_raw"]),
    "mainline_delay_clipped": sum(official_benchmark_history["mainline_delay_clipped"]),
    "upstream_boundary_delay": sum(official_benchmark_history["upstream_boundary_delay"]),

    "local_delay": sum(official_benchmark_history["local_delay"]),
    "fairness_penalty": sum(official_benchmark_history["fairness_penalty"]),
    "doorway_penalty": sum(official_benchmark_history["doorway_penalty"]),
    "safe_penalty": sum(official_benchmark_history["safe_penalty"]),
    "physical_penalty": sum(official_benchmark_history["physical_penalty"]),
    "spillback_penalty": sum(official_benchmark_history["spillback_penalty"]),
}

official_totals["capacity_penalty"] = (
    official_totals["doorway_penalty"]
    + official_totals["safe_penalty"]
    + official_totals["physical_penalty"]
    + official_totals["spillback_penalty"]
)

official_totals["raw_objective"] = (
    official_totals["mainline_delay"]
    + official_totals["local_delay"]
    + official_totals["fairness_penalty"]
    + official_totals["capacity_penalty"]
)

official_totals_df = pd.DataFrame([
    {
        "term": term,
        "value": value
    }
    for term, value in official_totals.items()
])

print("Official benchmark totals")
display(official_totals_df.round(6))


# 3. Ramp service / conservation metrics
R_final = official_benchmark_history["R_final"]
B_final = official_benchmark_history["B_final"]

official_service_metrics = {
    "initial_physical_ramp_queue_R": sum(
        float(ramp_queue_0[ramp])
        for ramp in ramp_ids
    ),

    "initial_external_spillback_queue_B": sum(
        float(external_queue_0_export[ramp])
        for ramp in ramp_ids
    ),

    "total_ramp_arrivals": sum(
        float(ramp_arrival_series[ramp][step])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),

    "total_requested_release": sum(
        float(official_benchmark_history["requested_release"][step][ramp])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),

    "total_actual_release": sum(
        float(official_benchmark_history["actual_release"][step][ramp])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),

    "final_physical_ramp_queue_R": sum(
        float(R_final[ramp])
        for ramp in ramp_ids
    ),

    "final_external_spillback_queue_B": sum(
        float(B_final[ramp])
        for ramp in ramp_ids
    ),

    "final_upstream_boundary_queue": float(
        official_benchmark_history["upstream_boundary_queue_final"]
    ),
}

official_service_metrics["ramp_mass_residual"] = (
    official_service_metrics["initial_physical_ramp_queue_R"]
    + official_service_metrics["initial_external_spillback_queue_B"]
    + official_service_metrics["total_ramp_arrivals"]
    - official_service_metrics["total_actual_release"]
    - official_service_metrics["final_physical_ramp_queue_R"]
    - official_service_metrics["final_external_spillback_queue_B"]
)

official_total_ramp_demand_to_account = (
    official_service_metrics["initial_physical_ramp_queue_R"]
    + official_service_metrics["initial_external_spillback_queue_B"]
    + official_service_metrics["total_ramp_arrivals"]
)

official_service_metrics["total_ramp_demand_to_account"] = (
    official_total_ramp_demand_to_account
)

if official_total_ramp_demand_to_account > 0:
    official_service_metrics["served_fraction"] = (
        official_service_metrics["total_actual_release"]
        / official_total_ramp_demand_to_account
    )
else:
    official_service_metrics["served_fraction"] = 1.0

official_service_metrics_df = pd.DataFrame([
    {
        "metric": metric,
        "value": value
    }
    for metric, value in official_service_metrics.items()
])

print("Official benchmark ramp service / conservation metrics")
display(official_service_metrics_df.round(6))


# 4. Validation checks that must pass before moving on.
def compute_mainline_mass_residual(history):
    initial_mainline = sum(
        float(mainline_initial_state[cell])
        for cell in cell_ids
    )

    final_mainline = sum(
        float(history["x_final"][cell])
        for cell in cell_ids
    )

    accepted_boundary = sum(
        float(history["q_in"][step]["Cell 1"])
        for step in range(num_steps)
    )

    external_inflow_total = sum(
        float(history["external_inflow"][step][cell])
        for step in range(num_steps)
        for cell in cell_ids
    )

    controlled_ramp_inflow_total = sum(
        float(history["controllable_onramp_in"][step][cell])
        for step in range(num_steps)
        for cell in cell_ids
    )

    terminal_mainline_out = sum(
        float(history["q_out"][step]["Cell 9"])
        for step in range(num_steps)
    )

    local_exit_total = sum(
        float(history["actual_f_out"][step][cell])
        for step in range(num_steps)
        for cell in cell_ids
    )

    return (
        initial_mainline
        + accepted_boundary
        + external_inflow_total
        + controlled_ramp_inflow_total
        - terminal_mainline_out
        - local_exit_total
        - final_mainline
    )


def compute_upstream_boundary_mass_residual(history):
    upstream_initial = 0.0

    total_boundary_demand = sum(
        float(q_in_boundary_series[step])
        for step in range(num_steps)
    )

    accepted_boundary = sum(
        float(history["q_in"][step]["Cell 1"])
        for step in range(num_steps)
    )

    upstream_final = float(
        history["upstream_boundary_queue_final"]
    )

    return (
        upstream_initial
        + total_boundary_demand
        - accepted_boundary
        - upstream_final
    )


def compute_max_receiving_violation(history):
    max_violation = 0.0

    for step in range(num_steps):
        for cell in cell_ids:
            pressure = (
                float(history["q_in"][step][cell])
                + float(history["external_inflow"][step][cell])
                + float(history["controllable_onramp_in"][step][cell])
            )

            violation = pressure - float(history["receiving"][step][cell])
            max_violation = max(max_violation, violation)

    return max(0.0, max_violation)


def compute_max_merge_violation(history):
    max_violation = 0.0

    for step in range(num_steps):
        cell9_pressure_from_merge = (
            float(history["q_out"][step]["Cell 8"])
            + float(history["actual_release"][step][merge_ramp_id])
            + float(history["external_inflow"][step]["Cell 9"])
        )

        violation = (
            cell9_pressure_from_merge
            - float(history["receiving"][step]["Cell 9"])
        )

        max_violation = max(max_violation, violation)

    return max(0.0, max_violation)


def compute_max_controlled_ramp_leakage_into_external_inflow():
    max_error = 0.0

    for step in range(num_steps):
        for cell in cell_ids:
            expected_external = float(
                undetected_entry_per_15sec_by_cell[cell]
            )

            actual_external = float(
                external_inflow_series[step][cell]
            )

            max_error = max(
                max_error,
                abs(actual_external - expected_external)
            )

    return max_error


normal_zero_commanded_release_series = {
    ramp: [
        float(commanded_release_series[ramp][step])
        for step in range(num_steps)
    ]
    for ramp in ramp_ids
}

for ramp in ["u_4th", "u_price", "u_mattie"]:
    normal_zero_commanded_release_series[ramp] = [
        0.0
        for _ in range(num_steps)
    ]

normal_zero_history = simulate_state_based_benchmark_480_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,
    q_in_boundary_series=q_in_boundary_series,
    commanded_release_series=normal_zero_commanded_release_series,
    ramp_arrival_series=ramp_arrival_series,
    external_inflow_series=external_inflow_series,
    fixed_outflow_series=fixed_outflow_series,
    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,
    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,
    tt_ff_min=tt_ff_min,
    delta_t=delta_t,
    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4,
    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    exit_split_by_cell=exit_split_by_cell,
    external_queue_0=external_queue_0_export,
    use_clipped_mainline_delay=use_clipped_mainline_delay
)

max_normal_ramp_control_state_delta = max(
    abs(
        float(official_benchmark_history["x"][step][cell])
        - float(normal_zero_history["x"][step][cell])
    )
    for step in range(num_steps)
    for cell in cell_ids
)

validation_metrics = {
    "mainline_mass_residual": compute_mainline_mass_residual(official_benchmark_history),
    "upstream_boundary_mass_residual": compute_upstream_boundary_mass_residual(official_benchmark_history),
    "ramp_mass_residual": official_service_metrics["ramp_mass_residual"],
    "max_receiving_violation": compute_max_receiving_violation(official_benchmark_history),
    "max_avila_merge_violation": compute_max_merge_violation(official_benchmark_history),
    "max_controlled_ramp_leakage_into_external_inflow": compute_max_controlled_ramp_leakage_into_external_inflow(),
    "max_normal_ramp_control_state_delta_when_zeroed": max_normal_ramp_control_state_delta,
}

validation_df = pd.DataFrame([
    {
        "check": check,
        "value": value
    }
    for check, value in validation_metrics.items()
])

print("Validation metrics")
display(validation_df.round(12))

if abs(validation_metrics["mainline_mass_residual"]) > 1e-8:
    raise AssertionError("Mainline mass conservation failed.")

if abs(validation_metrics["upstream_boundary_mass_residual"]) > 1e-8:
    raise AssertionError("Upstream boundary mass conservation failed.")

if abs(validation_metrics["ramp_mass_residual"]) > 1e-8:
    raise AssertionError("Ramp mass conservation failed.")

if validation_metrics["max_avila_merge_violation"] > 1e-8:
    raise AssertionError("Avila merge receiving constraint failed.")

if validation_metrics["max_controlled_ramp_leakage_into_external_inflow"] > 1e-12:
    raise AssertionError("Controlled ramps leaked into external inflow.")

if validation_metrics["max_normal_ramp_control_state_delta_when_zeroed"] <= 1e-9:
    raise AssertionError("Normal ramp commands still do not affect the mainline state.")

print("PASS: corrected benchmark validation checks passed.")


# 5. Normalization denominators
normalization_scales = {
    "mainline_delay_veh_min": 10000.0,
    "local_delay_veh_min": 1000.0,
    "fairness_penalty_units": 100.0,
    "doorway_penalty_units": 1000.0,
    "safe_penalty_units": 1000.0,
    "physical_penalty_units": 1000.0,
    "spillback_penalty_units": 1000.0,
}

benchmark_denominators = {
    "D_main_base": normalization_scales["mainline_delay_veh_min"],
    "D_local_base": normalization_scales["local_delay_veh_min"],
    "L_fair_base": normalization_scales["fairness_penalty_units"],
    "P_door_base": normalization_scales["doorway_penalty_units"],
    "P_safe_base": normalization_scales["safe_penalty_units"],
    "P_phys_base": normalization_scales["physical_penalty_units"],
    "P_spill_base": normalization_scales["spillback_penalty_units"],
}

for key, value in benchmark_denominators.items():
    if value <= 0:
        raise ValueError(f"{key} must be positive.")

benchmark_denominators_df = pd.DataFrame([
    {
        "denominator": denominator,
        "value": value
    }
    for denominator, value in benchmark_denominators.items()
])

print("Benchmark normalization denominators")
display(benchmark_denominators_df.round(6))


# 6. Build shared source-of-truth dictionary
shared_benchmark_inputs = {
    # core simulation settings
    "num_steps": num_steps,
    "delta_t": delta_t,
    "arrival_multiplier": arrival_multiplier,
    "use_clipped_mainline_delay": use_clipped_mainline_delay,

    "benchmark_profile_type": benchmark_profile_type,
    "benchmark_date": benchmark_date,
    "benchmark_source_file_requested": benchmark_source_file_requested,
    "benchmark_source_file": benchmark_source_file,
    "benchmark_start_hour": benchmark_start_hour,
    "benchmark_end_hour": benchmark_end_hour,
    "benchmark_start_time": benchmark_start_time,
    "benchmark_end_time": benchmark_end_time,
    "benchmark_window_label": benchmark_window_label,

    # objective weights / penalty coefficients
    "gamma": gamma,
    "lambda_1": lambda_1,
    "lambda_2": lambda_2,
    "lambda_3": lambda_3,
    "lambda_4": lambda_4,

    # CTM initial states
    "mainline_initial_state": mainline_initial_state,
    "ramp_queue_0": ramp_queue_0,
    "external_queue_0": external_queue_0_export,

    # benchmark time series
    "q_in_boundary_series": q_in_boundary_series,
    "observed_release_series": observed_release_series,
    "commanded_release_series": commanded_release_series,
    "ramp_arrival_series": ramp_arrival_series,
    "observed_offramp_series": observed_offramp_series,
    "single_day_benchmark_profile": single_day_benchmark_profile,

    "external_inflow_series": external_inflow_series,
    "fixed_outflow_series": fixed_outflow_series,
    "u_in_series": external_inflow_series,
    "f_out_series": fixed_outflow_series,
    "u_in_series_semantics": "legacy alias for external_inflow_series; controlled ramps are excluded",
    "f_out_series_semantics": "legacy alias for fixed_outflow_series",

    "benchmark_balance_audit_df": benchmark_balance_audit_df,
    "lateral_balance_calibration_df": lateral_balance_calibration_df,
    "free_flow_balance_audit_df": free_flow_balance_audit_df,
    "exit_split_by_cell": exit_split_by_cell,
    "exit_split_df": exit_split_df,
    "undetected_entry_vph_by_cell": undetected_entry_vph_by_cell,
    "undetected_entry_per_15sec_by_cell": undetected_entry_per_15sec_by_cell,
    "detected_offramp_cell_map": detected_offramp_cell_map,

    "generic_ramp_cell_map": generic_ramp_cell_map,
    "merge_ramp_id": merge_ramp_id,
    "merge_ramp_cell": merge_ramp_cell,

    "CAP9_VPH": CAP9_VPH,
    "BETA_SCALE": BETA_SCALE,
    "ENTRY_SCALE": ENTRY_SCALE,
    "MERGE_PRIORITY": MERGE_PRIORITY,

    "raw_exit_split_by_cell": raw_exit_split_by_cell,
    "raw_undetected_entry_vph_by_cell": raw_undetected_entry_vph_by_cell,

    # validation
    "validation_metrics": validation_metrics,

    # normalization
    "normalization_scales": normalization_scales,
    "benchmark_denominators": benchmark_denominators,

    # CTM parameters
    "backward_wave_speed_mph": backward_wave_speed_mph,
    "inflow_capacity": ctm_inflow_capacity_official,
    "outflow_capacity": ctm_outflow_capacity_official,
    "doorway_capacity_legacy": ctm_doorway_capacity_official,
    "physical_capacity": physical_capacity,
    "safe_threshold_capacity": safe_threshold_capacity,
    "movement_factor_by_cell": movement_factor_by_cell_official,
    "wave_speed_ratio_by_cell": wave_speed_ratio_by_cell_official,

    # ramp metadata
    "ramp_ids": ramp_ids,
    "ramp_cell_map": ramp_cell_map,
    "ramp_name_map": ramp_name_map,
    "ramp_max_queue_by_u": ramp_max_queue_by_u,
    "ramp_max_queue_named": ramp_max_queue_named,

    # free-flow travel time
    "tt_ff_min": tt_ff_min,

    # official benchmark results
    "official_benchmark_history": official_benchmark_history,
    "official_totals": official_totals,
    "official_service_metrics": official_service_metrics,
}


# 7. Save pickle file
with open("shared_benchmark_inputs.pkl", "wb") as fh:
    pickle.dump(shared_benchmark_inputs, fh)

print("Saved shared_benchmark_inputs.pkl")


# 8. Reload and verify export consistency
with open("shared_benchmark_inputs.pkl", "rb") as fh:
    shared_check = pickle.load(fh)

max_total_error = max(
    abs(
        float(shared_check["official_totals"][key])
        - float(official_totals[key])
    )
    for key in official_totals
)

max_denominator_error = max(
    abs(
        float(shared_check["benchmark_denominators"][key])
        - float(benchmark_denominators[key])
    )
    for key in benchmark_denominators
)

max_service_error = max(
    abs(
        float(shared_check["official_service_metrics"][key])
        - float(official_service_metrics[key])
    )
    for key in official_service_metrics
)

max_validation_error = max(
    abs(
        float(shared_check["validation_metrics"][key])
        - float(validation_metrics[key])
    )
    for key in validation_metrics
)

print("Max official total export mismatch:", max_total_error)
print("Max denominator export mismatch:", max_denominator_error)
print("Max service metric export mismatch:", max_service_error)
print("Max validation metric export mismatch:", max_validation_error)

if (
    max_total_error < 1e-9
    and max_denominator_error < 1e-9
    and max_service_error < 1e-9
    and max_validation_error < 1e-9
):
    print("PASS: shared_benchmark_inputs.pkl matches current official benchmark.")
else:
    raise AssertionError("shared_benchmark_inputs.pkl does not match current benchmark values.")


Official benchmark totals


,term,value
0,mainline_delay,2.574420e+04
1,mainline_delay_raw,2.554370e+04
2,mainline_delay_clipped,2.574420e+04
3,upstream_boundary_delay,0.000000e+00
4,local_delay,4.480673e+04
5,fairness_penalty,1.436237e+02
6,doorway_penalty,0.000000e+00
7,safe_penalty,0.000000e+00
8,physical_penalty,0.000000e+00
9,spillback_penalty,7.976926e+06


Official benchmark ramp service / conservation metrics


,metric,value
0,initial_physical_ramp_queue_R,0.000000
1,initial_external_spillback_queue_B,0.000000
2,total_ramp_arrivals,4387.200000
3,total_requested_release,3656.000000
4,total_actual_release,3649.200159
5,final_physical_ramp_queue_R,146.257223
6,final_external_spillback_queue_B,591.742619
7,final_upstream_boundary_queue,0.000000
8,ramp_mass_residual,-0.000000
9,total_ramp_demand_to_account,4387.200000


Validation metrics


,check,value
0,mainline_mass_residual,-0.000000e+00
1,upstream_boundary_mass_residual,0.000000e+00
2,ramp_mass_residual,-1.000000e-12
3,max_receiving_violation,0.000000e+00
4,max_avila_merge_violation,0.000000e+00
5,max_controlled_ramp_leakage_into_external_inflow,0.000000e+00
6,max_normal_ramp_control_state_delta_when_zeroed,2.837871e+02


PASS: corrected benchmark validation checks passed.
Benchmark normalization denominators


,denominator,value
0,D_main_base,10000.0
1,D_local_base,1000.0
2,L_fair_base,100.0
3,P_door_base,1000.0
4,P_safe_base,1000.0
5,P_phys_base,1000.0
6,P_spill_base,1000.0


Saved shared_benchmark_inputs.pkl
Max official total export mismatch: 0.0
Max denominator export mismatch: 0.0
Max service metric export mismatch: 0.0
Max validation metric export mismatch: 0.0
PASS: shared_benchmark_inputs.pkl matches current official benchmark.


In [8]:
# Minimal summary table
summary_df = pd.DataFrame([{"metric": k, "value": v} for k, v in official_totals.items()])
service_df = pd.DataFrame([{"metric": k, "value": v} for k, v in official_service_metrics.items()])
validation_summary_df = pd.DataFrame([{"metric": k, "value": v} for k, v in validation_metrics.items()])

print("Corrected compact benchmark complete.")
print("arrival_multiplier =", arrival_multiplier)
print("CAP9_VPH =", round(float(CAP9_VPH ), 3))
print("raw_objective =", round(float(official_totals["raw_objective"]), 6))
print("max_normal_ramp_control_state_delta_when_zeroed =", round(float(validation_metrics["max_normal_ramp_control_state_delta_when_zeroed"]), 12))
display(summary_df.round(6))
display(service_df.round(6))
display(validation_summary_df.round(12))


Corrected compact benchmark complete.
arrival_multiplier = 1.2
CAP9_VPH = 2400.0
raw_objective = 8047620.603495
max_normal_ramp_control_state_delta_when_zeroed = 283.787050324536


,metric,value
0,mainline_delay,2.574420e+04
1,mainline_delay_raw,2.554370e+04
2,mainline_delay_clipped,2.574420e+04
3,upstream_boundary_delay,0.000000e+00
4,local_delay,4.480673e+04
5,fairness_penalty,1.436237e+02
6,doorway_penalty,0.000000e+00
7,safe_penalty,0.000000e+00
8,physical_penalty,0.000000e+00
9,spillback_penalty,7.976926e+06


,metric,value
0,initial_physical_ramp_queue_R,0.000000
1,initial_external_spillback_queue_B,0.000000
2,total_ramp_arrivals,4387.200000
3,total_requested_release,3656.000000
4,total_actual_release,3649.200159
5,final_physical_ramp_queue_R,146.257223
6,final_external_spillback_queue_B,591.742619
7,final_upstream_boundary_queue,0.000000
8,ramp_mass_residual,-0.000000
9,total_ramp_demand_to_account,4387.200000


,metric,value
0,mainline_mass_residual,-0.000000e+00
1,upstream_boundary_mass_residual,0.000000e+00
2,ramp_mass_residual,-1.000000e-12
3,max_receiving_violation,0.000000e+00
4,max_avila_merge_violation,0.000000e+00
5,max_controlled_ramp_leakage_into_external_inflow,0.000000e+00
6,max_normal_ramp_control_state_delta_when_zeroed,2.837871e+02
